# 12장 RNN이라는 발상과 그 한계

## 1. Word2vec을 넘어서

자 우리는 앞에서 신경망 알고리즘의 개요를 꽤 자세하게 배웠고, 이를 상대적으로 간단히 활용한 word2vec에 대해서도 배웠습니다. 그런데 신경망 알고리즘으로 언어 자료를 처리하여 만들 수 있는 결과는 word2vec이 보여준 것보다 훨씬 다양합니다. 이번 시간부터는 그 가능성에 대해 배워볼 것입니다. 

11장에서 살펴본 Word2vec은 상대적으로 간단한 문제를 해결하는 역시 상대적으로 간단한 신경망 모델을 훈련시켰습니다. 이때 문제는 <맥락 단어>를 주고 <중심 단어>를 예측하거나 <중심 단어>를 주고 <맥락 단어>를 예측하는 것이었습니다. 이를 해내는 모델을 훈련하는 과정에서, '어떤 단어가 제공하는 자기 주변 단어에 대한 정보'가 단어별로 집약되어 모델에 파라미터로 표현됩니다. 그것을 우리는 '단어들의 의미 관계를 반영한 각 단어의 벡터 표현'으로 활용할 수 있다고 봤습니다. 

이런 접근은 계산적인 관점에서 무척 효율적이고, 놀라울 정도로 많은 의미 정보를 포착하는데 성공했습니다. 하지만 동시에 꽤나 강한 제약과 정보 단순화를 동반합니다. 언어가 가진 정보의 상당 부분을 의도적으로 버리고 있습니다. 첫 번째, '주변'의 크기를 분석자가 의도적으로 제한합니다. 윈도우(window) 크기가 그거죠. 3이라고 설정하면, 전후의 3개 단어만 고려합니다. 그 정도 거리에서만 단어가 다른 단어 예측에 도움이 된다고 보는 것입니다. 두 번째, '순서'라는 정보를 고려하기 어렵습니다. 예를 들어 일단 '주변'이라고 판정되는 윈도우 안에만 들어오면, 그 안에 들어오는 단어들 사이에 특별한 차이를 두지 않습니다. '나는' '맛있게', '밥을' '먹다가', '잤다' 라는 문장이 있다면, '밥을'이라는 토큰과 '나는'이 맺는 관계는, '밥을'이 '먹다가'라는 토큰과 맺는 관계와 차이가 없습니다. 즉 word2vec이 운용하는 신경망 모델은 순서라는 정보와 먼 거리를 두고 일어나는 토큰의 상호작용 패턴을 반영하지 못합니다. 

그런데 실제로 언어가 어떤 의미나 효과를 만들어내는 방식은, word2vec 처럼 특정 거리에서 토큰 (혹은 단어)이 동시 출현하는 것에만 의존하지 않습니다. 토큰의 순서 역시 중요한 역할을 합니다. '어떤 남자가 닭을 먹었다'와 '어떤 닭이 남자를 먹었다'가 매우 다른 의미와 정서를 만들어내는 것처럼, 순서는 의미의 강도와 방향을 바꾸는 정보입니다. 멀리 떨어진 토큰 (혹은 단어)이 상호작용하는 경우도 없지 않습니다. '원광이는 자기 방 컴퓨터 의자에 앉아서 작업에 열중하고 있었다. 혜미가 들어와 그에게 턱짓으로 인사했다.'라는 문장이 있을 때, '그에게'가 구체적으로 무엇인지 알기 위해서는 '그에게' 근처에 나오는 단어가 아니라 꽤 떨어진 상태로 앞에 나오는 '원광이'를 참조해야 합니다. 그리고 이런건 흔한 언어 활동에 가깝죠. 그렇기에 다음과 같은 질문이 자연스럽게 나옵니다. word2vec을 넘어 언어 자료에서 이런 추가적인 정보를 활용할 수 있는 신경망 모델은 어떻게 만들 수 있을까요?

활용 정보의 종류만이 아닙니다. 모델의 구체적 기능 또한 word2vec보다 좀 더 다양했으면 합니다. 사실 word2vec이 운용하는 신경망 모델이 구체적으로 하는 일은 '단어의 예측'입니다. CBOW에서는 '중심 단어'를 예측하는 것이고, Skip-gram에서는 '맥락 단어'를 예측했습니다. 그런데 우리는 언어 정보를 활용하는 모델에게 시키고 싶은 일이 많습니다. 문장이나 문서를 특정한 범주로 분류하는 일도 시키고 싶을 수 있고 (예를 들어 리뷰에 담긴 정서를 분류하는거죠. 긍정 리뷰/부정리뷰 이런 식으로), 한글을 영어로 번역하는 일을 시키고 싶을 수도 있고, 질문에 적절히 응답하는 모델을 만들고 싶을 수도 있습니다. 이런 다양한 기능을 가능하게 하는 신경망 모델 디자인은 어떠해야 할까요?

Recurrent Neural Network (RNN)과 Transformers 는 이러한 한계를 극복하려는 시도 속에서 등장한 신경망/딥러닝 모델 아키텍쳐입니다. 이들은 word2vec 보다 언어 정보를 좀 더 다각도로 흡수하고 또 다양한 결과물을 낼 수 있도록 설계되었습니다. 이들의 성과는 매우 눈부셨고, 그 결과가 우리가 지금 누리고 있는 여러 언어 인공 지능들입니다. 기계 번역 서비스나 챗봇 등이 대표적이죠. 물론 이것이 Word2vec이 더 이상 쓸모가 없어진 모델이라는 뜻은 아닙니다. 우리의 목적이 '언어 자료의 정보를 좀 더 다양하게 활용하여 복잡하고 다양한 일을 해낸다'는 것이라면, 그 관점에서는 한계가 있다는 말입니다. 거꾸로 우리의 목적이 '효율적인 계산을 통해 단어의 의미적 거리를 반영한 단어의 수치적 표현을 얻는다'는 것이라면, 여전히 Word2vec은 무척 훌륭한 방법입니다. 여러 번 말씀드렸듯이 모델의 가치는 목적에 따라 결정되기 마련입니다. 다만 지금부터는 우리가 '언어 자료의 정보를 좀 더 다양하게 활용하여 복잡하고 다양한 일을 해낸다'는 목적을 가진다고 했을 때, 신경망/딥러닝 모델이 우리에게 주는 가능성을 살펴보도록 하겠습니다. 

## 2. RNN 의 등장

RNN(Recurrent Neural Network)은 위에서 언급한 '순서'와 '먼 거리 의존'을 신경망 구조 안에서 본격적으로 흡수하여 활용하려고 한 대표적인 초기 신경망 아키텍처라고 할 수 있을 것입니다. 사실 언어 자료 만을 염두에 두고 등장한 모델 아키텍쳐는 아니고 순서가 있는 데이터를 고려하기 위한 것이었는데, 언어 자료 처리에 많이 적용되었습니다. 

여기서 '아키텍처'라는 말은 쉽게 말해 신경망의 구조적 디자인을 뜻합니다. 즉 신경망 모델이 입력된 신호를 어떤 절차를 거쳐 어떻게 흘려보내는지에 대한 구조적 디자인이 존재할 수 있는데, 그 스타일이 여러 가지입니다. 기본적인 신경망 아이디어는 같아도, 이 구조가 어떠냐에 따라 전혀 다른 모델이 될 수 있습니다. 우리가 개요 강의에서 살펴본 것은 가장 기본적인 방식이고, 그 기반에서 여러 가지 다양한 방식이 만들어졌습니다. 

예를 들어 CNN (Convolutional Neural Network)은 주로 이미지 같은 데이터에 적용됩니다. 시각 데이터는 주로 행렬 형태로 존재합니다. (흑백 이미지면 행렬로 표현이 가능합니다. 컬러라면, 행렬 세 개로 표현이 가능하죠.) 10장에서 우리는 이것을 길게 펴서 일반적인 벡터로 처리했습니다. 그런데 CNN은 Convolution 함수라는 것을 동원하여 2차원 공간 정보를 유지한 상태에서 신경망 모델에 이미지 정보가 흐르게 합니다. 공간 구조 정보를 활용한 연산이 진행되므로, 이미지 관련 문제에서 좋은 성능을 보입니다. 이렇게 하려면 특수한 디자인이 필요한데, 그걸 CNN이라고 하는거죠. 

마찬가지로 RNN은 지금부터 설명할 특수한 신경망 구조 디자인이 있기에 따로 이름을 붙이는 것입니다. 개인적으로는 왜 쉬운 말인 '디자인'을 쓰지 않는 것인지 의아한데, 아마도 영어 단어가 가진 뉘앙스와 학문 전통에 따른 결정이었지 않을까 합니다. 다른 문헌과 일치시키기 위해 이 강의안에서도 아키텍처라고 부르겠습니다. 

자 그러면 RNN이 어떤 방식으로 신경망 모델을 만드는지 본격적으로 알아봅시다. 우선 언어자료가 '유의미한 순서를 가진 토큰의 연쇄'라는 점을 기억합시다. 앞선 강의에서 말했듯 토큰은 분석자가 설정하는 언어의 최소 단위입니다. 이런 토큰이 자동으로 주어지지는 않지만, 4강에서 소개한 토큰화를 위한 다양한 기법을 통해 토큰화가 되어 있다고 가정하겠습니다. 예를 들어 '안녕하십니까 여러분' 이라고 되어 있는 것이 아니라 ['안녕', '하십니까', '여러분'] 이렇게 되어 있다고 하겠다는 것입니다. 즉 언어 구성 요소가 순서를 가지고 나열되어 있다고 가정할 것입니다. 그래서 지금부터 하나의 자료를 하나의 시퀀스라고 부겠습니다. 그리고 '순서'를 '시점'이라고 부르겠습니다. 즉 첫 번째 토큰은 첫 번째 시점에 존재하고, 두 번째 토큰은 두 번째 시점에 존재하는 것입니다.

RNN은 간략히 말하면 시퀀스를 구성하는 토큰마다 연산이 가해지고, 그 결과를 그 시점에서 출력하는 동시에 다음 시점의 맥락 정보로 넘겨주는 형태의 모델 아키텍쳐를 의미합니다. 말로 설명하면 좀 복잡하니 아래 그림을 봅시다. 


<div style="text-align:center;">
    <img src="_static/figures/vanilla rnn 기본 도식.jpg" width="700">
</div>

아래에 t_0, t_1, t_2는 시점을 의미합니다. 그리고 X_0, X_1, X_2는 시점별로 존재하는 토큰을 의미하죠. X_0부터 살펴봅시다. 이게 RNN 블럭에 들어간다는 것은, 특정한 연산을 거쳐 뭔가 결과를 만들어낸다는 뜻입니다. 그게 H_0으로 출력됩니다. 즉 H_0가 이 시점의 핵심 결과물입니다. 그런데 이는 동시에 다음 시점 연산에 고려될 맥락이 됩니다. 그게 지금 다음 RNN 블럭으로 H_0가 보내지는 것으로 표현되어 있습니다. 

그래서 X_1이 H_1을 만들어내는 과정에서는 이 H_0도 역할을 하게 만듭니다. 그리고 계속 이렇게 이어지는거죠. **즉 기본적으로 RNN 블럭은 해당 시점 토큰 X와 과거 시점 H를 받아서 결과를 만듭니다. 그리고 그 결과가 해당 시점 출력값의 기반이 되는 동시에 다음 시점에 고려될 '맥락'이 되는거죠. 이 맥락을 우리는 은닉 상태 (hidden state) 라고 부릅니다.** 그런데 최초 시점인 t_0에는 전 시점에서 넘어올 맥락 정도/은닉 상태가 따로 없으므로 H_init을 따로 분석자가 설정해서 넣어줍니다. 가장 손쉽게는 차원은 맞추되 내용은 다 0으로 채운 영벡터를 넣어주는 방식이 있겠네요. 

우리가 앞에서 봤던 신경망 모델과는 많이 달라보이죠? 그런데 사실 거의 동일합니다. 다만 앞에서 소개한 신경망 모델이 매 입력 자료마다 붙어 있고, 그게 전 자료에서는 왼쪽->오른쪽 방향으로 그려져 있었는데, 그게 지금 아래 -> 위로 그려져 있는거죠. 예를 들어 X_0가 RNN 블럭에서 겪는 연산은, 우리가 앞에서 본 word2vec에서 토큰 별로 일어나는 연산과 동일합니다. 즉, 아래와 같은 그림 형태인 것이죠. 예를 들어 X_0가 길이 4짜리 벡터이고, RNN이 이를 길이 3인 벡터로 줄이도록 설계되어 있다면, 입력 벡터에 (3x4) 행렬을 곱하는 방식의 연산이 적용됩니다. 10장과 11장에서 봤던 그래프들이 왼쪽 -> 오른쪽으로 그려져 있었다면, 그게 지금 90도 회전되어 아래 -> 위로 되어 있는 셈이지요. 

<div style="text-align:center;">
        <img src="_static/figures/vanilla rnn 국소 확대 도식.jpg" width="700">
</div>

다만 11장에 나온 word2vec과 약간 다른 것은, word2vec에서는 원-핫 벡터가 들어갔는데, 통상적인 RNN 세팅에서는 토큰들이 적절한 벡터 표현으로 전환되었다고 가정하고 (이 경우는 [2,1,0,1] 이네요), 그 토큰을 표현하는 벡터가 들어간다고 보고 있는 점입니다. 즉 지금부터 X로 표현하는 것들은 토큰 자체라기보다, 이것이 임베딩된 결과, 즉 수치정보입니다. 좀 더 명시적으로 이런 기능을 하는 embedding layer도 표현할 수도 있겠지만, 간단하게 표현하기 위해 각 토큰들이 embedding을 거쳐 벡터 표현을 얻었다고 되었다고 가정하겠습니다. 

그래서 RNN 블럭에서 일어나는 연산을 좀 더 공식적으로 표현해보면 이렇습니다. 

$$
\begin{aligned}
H_t = tanh(W_x \cdot X_t + W_h \cdot H_{t-1} + b)
\end{aligned}
$$

$W_x \cdot X_t$ 파트는 제가 지금 그림으로 그려놓은 파트입니다. 그림에서 길이 4인 벡터가 3으로 줄어들었는데, 그게 (3x4) 행렬을 곱하는 것이라고 말씀드렸죠. 이 경우 그 (3x4)행렬이 $W_x$ 인 셈입니다. x에 곱해지는 가중치 행렬인 셈이지요. RNN은 이 결과를 바로 한 시점 전으로부터 전달된 은닉 상태에 적용된 연산의 결과와 더합니다. 그게 $W_h \cdot H_{t-1}$ 파트로 표현되고 있습니다. 그리고 둘이 더해지는 거죠. 그런 다음 편향이 더해집니다. 그런 다음 거기에 앞에서 배웠던 tanh 활성화 함수가 적용됩니다. 그것이 RNN 블럭의 기본적인 결과입니다. 

그런데 보통 결과는 Y라고 하는데 왜 H라고 하는걸까요? 아마도 이 출력물의 기본 역할 때문인 듯합니다. RNN 블럭의 산물이 다음 블럭으로 건너가는 은닉 상태가 되는 것은 확실한데, 해당 시점의 출력인 Y를 만들 때는 이 은닉 상태에 다시 한 번 추가적인 작업을 해서 원하는 형태를 만드는 경우가 대부분이거든요. 즉 $Y_t$는 여기서 한 번 더 작업을 넣어서 만드는거죠. 예를 들어 별도의 층을 통과시키는 방식으로 각 시점의 출력을 만들 수 있습니다. $Y_t = W_y \cdot H_t + b$ 이런 식이 예가 될 수 있겠네요. 

그런데 지금은 가장 단순한 형태로 설명하고 있고, 작업 목표마다 마지막 출력값을 위한 처리가 달라지는 경우가 많아 그림에서는 Y는 따로 표현하지 않았습니다. 다만 실제 RNN운용에서는 보통 각 시점의 H에 뭔가 추가적인 연산이 적용되곤 한다는 것만 염두에 두시기 바랍니다.  

여기서 꼭 기억해야 할 중요한 점은, **한 작업에 적용된 모든 RNN 블럭에서 $W_x$와 $W_h$와 b는 동일하다는 점입니다.** 즉 시점과 순서에 따라 입력값과 은닉 상태값이 달라지지만, 그들에게 적용되는 연산은 동일합니다. 그래서 이걸 '순환신경망' (recurrent neural network)이라고 부르고 아래와 같은 그림으로 표현하는 것입니다. **왜냐하면, 값이 들어가서 은닉 상태를 만든 다음 그걸 다시 동일한 연산 블럭으로 돌려주는 꼴이 거든요.** 

<div style="text-align:center;">
        <img src="_static/figures/vanilla rnn 축소 도식.jpg" width="700">
</div>

이게 RNN의 개요입니다. 여기서 좀 더 처치가 들어가는 경우가 많지만, 기본적인 모델 디자인의 핵심은 이것이라고 볼 수 있습니다. 그러면 이렇게 모델 디자인을 짜면 뭐가 좋은걸까요? 혹은 왜 이렇게 모델을 디자인 하는 걸까요? 

- 일단 이렇게 되면 시퀀스로 존재하는 자료 혹은 데이터의 '순서 정보'가 자연스럽게 연산에 반영됩니다. 지금 각 시점의 입력값 $X_t$은 단독으로 해당 시점의 결과를 산출하지 않습니다. 대신 그 전 시점의 입력값들이 종합된 결과인 $H_{t-1}$이 만들어내는 효과를 고려하여 결과를 산출합니다. 앞서 나온 수식이 그걸 선언하고 있는 거죠. 즉 지금 시점 결과는 과거 시점의 결과를 고려하여 결정된다는 말입니다. 예를 들어 3번째 시점의 출력은 3번째 입력값 뿐만 아니라 1,2번째 입력값이 종합된 결과가 함께 고려되어 만들어 집니다. 즉 매 시점의 연산이 그 전 시점에 존재한 자료와 그로부터 나온 정보를 고려하게 되고, 이게 곧 모델이 데이터가 가진 순서 정보를 반영하는 일이지요. 
  
- 이 때문에 먼 거리를 두고 떨어진 토큰들 사이의 상호작용도 원칙적으로 가능해집니다. 매우 매우 앞쪽에 있는 토큰의 정보가 뒤쪽으로 전달이 적어도 원리적으로는 가능하거든요. 예를 들어 1번째 토큰의 입력값은 중간 연산을 거쳐 17번째 토큰이 만들어낼 출력에 영향을 미칠 수 있습니다. (다만 이게 실질적으로 잘 안될 수 있습니다. 자세한 사항은 RNN의 한계 파트에서 살펴보지요)
  
- 이런 디자인은 텍스트 자료의 길이가 가변적이라는 특징에 대응하기 좋습니다. 텍스트 자료가 자료로서 가진 특징 중 하나는 사례마다 길이가 다르다는 것입니다. 우리에게 익숙한 자료는 길이가 일정한 경우가 많습니다. 예를 들어 설문조사 데이터는 매 케이스가 보통 같은 개수의 변수를 가집니다. 단순화해서 어떤 자료가 '성별', '소득', '직업'이라는 변수를 가졌다고 하면 모든 케이스는 3차원 벡터로 표현할 수 있습니다. [성별값, 소득값, 직업값]으로 표현되는 셈이죠. 이런 자료를 가지고 분류 모델을 만든다면, 모델은 이 3차원 벡터를 여러 개 받아서 학습하고 분류하는 기능을 하면 됩니다. 그런데 텍스트 자료는 케이스마다 길이가 다릅니다. 예를 들어 식당 리뷰가 긍정적인지 부정적인지 분류하는 모델을 만들고 싶다고 해보겠습니다. 실제 리뷰 사례는 이런 식일 겁니다. '사장님 친절하고 맛도 좋아요!', '너무 짜요', '숫가락에 고춧가루 묻어 있어서 기분 별로였음' / 이 세 가지 리뷰는 모두 길이가 다릅니다. 리뷰를 구성하는 토큰을 숫자나 벡터로 전환해서 이어 붙인다고 생각하면, 두 번째 리뷰는 짧지만, 세 번째 리뷰는 엄청 길죠. 이건 모델을 디자인 할 때 꽤 난감한 지점이 됩니다. 보통 우리가 배운 모델들은 주어지는 데이터의 길이가 일정할 것을 전제하는 경우가 많거든요. 그런데 이렇게 그때 그때 데이터의 길이가 달라지면 어쩌지 싶습니다. 그런데 RNN은 같은 RNN 블록을 텍스트 자료의 토큰 마다 적용합니다. 이렇게 되면, 그냥 토큰 시퀀스 길이에 맞춰서 RNN 블럭을 반복 적용하면 됩니다. 즉, 데이터 길이에 따라 유연하게 대처할 수 있는 능력이 생긴다는 말입니다.
  
- 마지막으로 RNN 같은 디자인은 파라미터 숫자 증가를 억제하는 효과를 가집니다. 이건 현실적으로 학습 가능한 모델을 만드는 핵심적 장점입니다. 텍스트 자료가 가지는 또 하나의 특징은, 대체적으로 엄청 길다는 것입니다. 앞서 말한 리뷰, 즉 댓글은 텍스트 자료 중 그래도 짧은 편인데, 아무리 짧아도 보통 10개-15개의 토큰은 가집니다. '이 과목 강사인 조원광은 말이 엄청 많은데 재미는 별로 없어요'라는 강의 평가가 있다고 하면, 이 간단한 평가만 해도 10개 토큰으로 이루어져 있죠. 그런데 우리가 Word2vec 절에서 봤듯이 개별 토큰들은 작게는 수십, 많게는 수천 차원의 벡터로 전환되곤 합니다. 적당히 100차원으로 전환한다고 하고, 이걸 한 방에 입력하기 위해 그냥 이어붙인다고 한다면, 이 리뷰는 100차원 벡터를 10개 이어붙인, 즉 1000차원 벡터로 전환될 것입니다. 이 자료에 affine layer를 적용해서 입력값을 1000차원의 은닉 상태로 전환한다고 합시다. 그러면 그 층에서 가중치 행렬만 (1000, 1000) 행렬이 됩니다. 100만 개의 숫자로 이루어진 행렬이 필요하다는 말이지요. 요컨대, 텍스트 자료는 쉽게 길어질 수 있습니다. 그걸 벡터로 표현하면 엄청 높은 차원의 벡터가 됩니다. 이걸 기본적인 신경망, 예를 들어 완전 연결 신경망 같은 것으로 대응하려고 하면, 파라미터가 엄청 많아집니다. 이건 쉽게 말해 모델의 조정점이 너무 많아진다는 것이고, 모델 학습에 부담을 줍니다. 그런데 RNN은, 동일한 파라미터를 공유하는 RNN 블럭을 매 토큰 마다 반복 적용합니다. 이건 가변적인 길이에 대응할 수 있는 디자인일 뿐 아니라, 길이가 길어져도 파라미터 숫자가 특별히 늘어나지 않을 수 있는 디자인입니다. 조금 수학적으로 표현하면 이렇습니다. 토큰 하나를 d차원으로 표현한다고 하겠습니다. 그리고 길이가 T라고 하죠. 이걸 그냥 서로 이어붙여서 완전연결층으로 처리하려고 하면, 입력 차원이 Td 차원이 됩니다. 이걸 h 차원으로 변형한다고 하면 가중치 행렬만 (h, Td)가 됩니다. 여기만 Tdh개의 파라미터가 필요하죠. 그런데 RNN은 그냥 (h, d)라는 동일한 행렬을 각 토큰에 적용합니다. 물론 이거말고 전 시점 은닉 상태를 처리하는 행렬도 필요하지만, 핵심은 T가 늘어난다고 파라미터 숫자가 그에 따라서 늘어나지 않는다는 것입니다. 파라미터 숫자를 이렇게 억제할 수 있는 것은, 실제로 학습 가능한 모델을 만들 수 있는 핵심적 특징입니다. (여담이지만, 앞서 언급한 CNN도 이런 '파라미터를 공유하는 하위 구조'를 이미지의 각 부분에 적용하는 방식입니다.)

RNN이 실제로 어떤 다양한 문제 해결에 활용되는지 살펴보시면 이런 모델 디자인의 목적과 장점에 대해 좀 더 감이 오실겁니다. 

우선 RNN은 순서를 고려해야 하는 시퀀스 라벨링(Sequence labeling) 문제에 적합한 면이 있습니다. 시퀀스 라벨링 문제는 입력이 시퀀스인데, 시퀀스를 구성하는 구성요소들에 특정한 범주를 부여하는 문제를 의미합니다. 예를 들어 앞에서 우리가 배웠던 '품사 부여' 문제를 생각해보죠. 언어가 시퀀스 데이터인데, 그 구성 요소인 각 형태소에 품사를 부여해야 합니다. 그런데 이걸 위해서는 순서를 고려하는 것이 필수적입니다. 품사의 부여가 순서에 영향을 받거든요. 예를 들어 영어를 기준으로 하면 '형용사' 뒤에는 '명사'가 올 확률이 다른 품사보다 훨씬 크죠. 시적 허용까지 고려하면 '형용사' 뒤에 '부사'가 올 수도 있지만, '형용사' 뒤에는 '명사'가 올 가능성이 훨씬 큽니다. 그러니 이런 순서가 주는 제약을 고려하면 훨씬 성과가 좋죠. 그리고 RNN은 이런 작업에 적합한 디자인을 제공합니다. 

언어 모델 (language model)을 만드는 것도 이런 RNN과 잘 어울리는 대표적인 문제입니다. 언어 모델은 기본적으로 토큰 혹은 단어 연쇄에 확률을 부여하는 모델을 의미합니다. 즉 예를 들어서 '오늘' '날씨가' '좋아서' '테라스에서' '차를' 까지 토큰의 연쇄가 주어졌다고 하죠. 그 다음에 들어오기 가장 적절한 토큰은 무엇일까요? 이를 위해서는 후보가 되는 토큰들에 확률을 부여해주는 모델이 필요합니다. 즉 $P(x_5|x_0 = 오늘, x_1 = 날씨가, x_2 = 좋아서, x_3 = 테라스에서, x_4 = 차를)$을 주는 모델이 필요하단 말입니다. 이런걸 하는게 언어 모델입니다. 언어 모델의 성능이 좋을 경우 언어 모델은 '마셨다'나 '먹었다'에 높은 확률을 부여할 것입니다. '기도했다' 같은 토큰에는 낮은 확률을 부여하고요. 

그런데 이런 언어 모델을 만드는데 RNN이 잘 어울립니다. 순서가 모델 디자인 때문에 자연히 고려되고, 매 시점의 출력은 그보다 과거에 존재했던 입력들을 종합하게 만들고 있으니까요. 우리가 살펴본 RNN의 기본 모형에 $H_t$에 선형 변환을 거쳐 소프트맥스를 걸면, 바로 언어모델을 훈련할 수 있는 형태가 됩니다. 또한 앞에서 살펴본 가변 길이 대응 능력과 파라미터 공유 구조 덕분에, 학습 가능한 규모의 모델을 만들기 좋습니다. 실제로 초기 언어 모델은 RNN을 활용한 경우가 아주 많았습니다. 

간단히 그림으로 표현하면 아래와 같은 식입니다. 

<div style="text-align:center;">
        <img src="_static/figures/vanilla rnn 언어 모델 훈련 예시.jpg" width="700">
</div>

그림은 '나는 밥을 먹어'라는 문장이 있으면, 모델이 지금까지 입력된 토큰에 근거하여 다음을 예측하게 만드는 모델을 RNN으로 훈련시키는 사례입니다. <시작> <끝> 같은게 뭔가 싶으실텐데, 실제 언어 자료 활용에서는 <시작> <끝> 같은, 자료의 시작과 끝을 알리는 특수 토큰을 포함하는 경우가 많습니다. 

이 학습의 목표는 간단합니다. <시작>이 들어가면 '나는'을, '나는'이 들어가면 '밥을'을, '밥을'이 들어가면 '먹어'를 예측하게 만들고 싶은거죠. 그리고 토큰이 순서를 갖춰서 들어가면서 앞에 미리 들어가 있는 토큰이 처리되어 넘어온 정보를 잘 활용하도록 하는 것입니다. 이걸 하기 위해 RNN에 토큰을 순서대로 입력하고, 바로 다음 토큰을 예측하도록 훈련합니다. 매 시점마다 $H_t$가 생산될 텐데, 그걸 적절히 변형하여 단어 예측에 사용합니다. 예를 들어 완전연결층을 통해 어휘 종류 만큼의 차원을 가진 벡터로 전환하고 거기에 소프트맥스를 통과시키면 단어의 확률 분포 형태가 될 수 있습니다. 모델이 주는 확률 분포와 실제 정답을 가지고 손실을 계산할 수 있습니다. 여기서 하늘색 박스는 $H_t$를 이런 방식으로 변형하는 과정을 의미하고, 보라색 박스는 목표로 맞춰야 하는 토큰을 의미합니다. 

이렇게 설정한 후 우리의 모델이 실제 정답을 잘 맞추도록, 즉 손실이 줄도록 파라미터들을 잘 조절하게 만드는거죠. 앞서 말씀드린 것처럼 이런 훈련은 모델이 가변 길이 데이터를 잘 처리하면서 순서 정보를 활용할 수 있어야 하기에, 이런 RNN 기반 디자인이 적합한 면이 있는 것입니다. 

이런 방식 말고도 여러 가지가 가능합니다. 위에서 말한 두 가지 예는 '매 시점의 출력을 모두 활용하는' 사례 였습니다. 하지만 꼭 그래야 할 필요는 없습니다. 그렇지 않게 활용하는 방식이, '마지막 출력'만 활용하는 방식입니다. RNN의 연산 구조를 보시면, 시퀀스가 순서대로 정보를 누적해갑니다. 마지막 출력값은 해당 시점보다 앞에 있었던 입력을 모두 압축한 형태입니다. 그것도 순서 고려한 방식으로요. 만약 우리가 '어떤 시퀀스 데이터 전체에 근거한 분류나 예측을 하려는데, 그 시퀀스 데이터가 가진 순서를 고려하고 싶다'는 상황이라면, 이 마지막만 활용하는것도 방법입니다. 전형적인 예가 문장이나 짧은 글의 감정 분류 문제입니다. '이 병원은 의사는 불친절한데 간호사가 무척 친절해요.' 라는 시퀀스가 있다고 하죠. 이것의 감정을 긍정/부정으로 분류해야 한다고 합시다. 그러면 마지막 입력까지 모두 입력된 상태에서 만들어지는 은닉 상태를 (즉 시퀀스 마지막 시점이 t면, $H_t$) 활용해서 분류 모델을 만드는 것이 합리적입니다. 그때의 산출은 앞에서 입력된 자료가 모두 순서를 고려하여 종합된 상태니까요. 그러면 원리적으로 '이 병원은 의사는 친절한데 간호사가 무척 불친절해요'라고 말하는 것과의 차이도 잡힙니다. 늘 그런건 아니지만 보통 우리는 강조하고 싶은걸 마지막에 놓으니까, 전자는 전반적으로 좋은 경험을 후자는 전반적으로 나쁜 경험을 말하려고 할 공산이 크죠. RNN처럼 순서를 고려하는 모델의 마지막 값에 근거한 연산은 이런 순서 차이도 이론적으로 고려할수 있습니다. 

## 3. RNN의 첫 번째 한계: 정보의 중첩과 약화

여기까지 보면 의아합니다. 이렇게 좋은 모델을 왜 요즘은 잘 안쓸까요? RNN은 트랜스포머가 최근 빠른 속도로 대체하고 있고, 사실상 거의 대체되었다고 판단하는게 오히려 합당할 듯합니다. 2017년 제안된 트랜스포머 아키텍쳐는 시퀀스 데이터를 처리하는 새로운 표준이 되어 가고 있는 상황입니다. RNN에서 뭔가 모자란 점이 있었고 그걸 트랜스포머가 훌륭하게 해결했기에 대체되었을텐데, 그러면 모자란점이 뭐였을까요? 

RNN의 한계를 추상적 언어를 통해 정리하자면 크게 두 가지라고 할 수 있습니다. 1) 정보의 중첩과 약화, 2) 순서를 반영한 RNN 연산의 기계적 비효율. 

일단 3절에서는 첫 번째인 정보의 중첩과 약화에 집중하여 살펴봅시다. 이건 길이가 긴 시퀀스를 타고 정보가 흐르면서 초기에 입력되고 계산된 정보가 그 영향력을 점진적으로 상실하여 모델에서 역할을 하지 못하는 현상입니다. 그리고 이건 언어 작용에서 구성요소들이 꽤 거리를 둔 상태에서 서로 연결되어 작동하는 경우가 드물지 않은 면을 고려할 때 중요한 문제입니다. 

기본적으로 RNN이 딥러닝 아키텍쳐라는 점을 염두에 두고, 순전파 - 즉 입력 정보가 처리되어 출력으로 이어지는 과정과, 역전파 - 손실에 기반하여 각 파라미터의 그레디언트가 계산되는 과정인 둘로 나눠서 검토해봅시다. 

우선 순전파인 상황을 보죠. 우리가 원하는건 RNN 모델이 시퀀스 초기에 들어갔던 입력도 잘 고려해서 출력을 만드는 것입니다. 앞서 예로 들었던 문장을 약간 변형해서 이런 문제를 만든다고 합시다. '원광이는 자기 방 컴퓨터 의자에 앉아서 작업에 열중하고 있었다. 혜미가 들어와 ??에게 턱짓으로 인사했다.' 여기서 ??을 제대로 예측하려면, 맨 앞에서 입력된 '원광이'가 계속 살아서 나중에 ??를 출력할 때 고려되어야 합니다. 

그런데 이게 쉽지 않습니다. **정보가 약화되고 중첩**됩니다. 연산이 거듭될 수록, 앞쪽에서 입력된 정보의 영향력이 줄어들기 쉽습니다. 

RNN 연산 구조를 떠올려 봅시다. 어떤 입력이 들어가면 어떤 과정을 겪게 되나요? $X_t$가 입력되었다고 가정합시다. 우선 $W_x$라는 가중치 행렬이 곱해집니다. 그리고 은닉층에서 온 결과인 $W_h \cdot H_{t-1}$와 더해져서 tanh 함수에 들어갑니다. 이 결과는 $H_t$가 되어서 다음 시퀀스에서 연산에 반영됩니다. 즉 아래와 같이 계속되는건데, $X_t$의 정보는 이후 은닉 상태 (두 번째 수식의 경우 $H_t$)에 반영되어 전달되겠죠. 


$$
\begin{gathered}
H_t = tanh(W_x \cdot X_t + W_h \cdot H_{t-1} + b)\\[15pt]
H_{t+1} = tanh(W_x \cdot X_{t+1} + W_h \cdot H_{t} + b)\\
...
\end{gathered}
$$

이 구조에서 $X_t$가 연산을 거듭하면서 어떻게 변화하는지 잘 생각해봅시다. $X_t$에게 일어나는 일로만 한정해서 수식을 바라보면, 아마 이런 식일 것입니다. 

$$
\begin{gathered}
\tanh\Big(W_h \cdot \tanh\big(W_h \cdot \tanh(W_x \cdot X_t)\big)\Big)....
\end{gathered}
$$

즉 처음에는 $W_x$가 곱해져서 tanh에 들어가고, 이후에는 은닉 상태의 부분이 되어 계속 $W_h$가 곱해지는 일과 tanh를 통과하는 일이 반복되지요. 이런 식이기에, $X_t$가 가졌던 원래 신호는 시퀀스가 진행되면서 약해지고 흐려지기 쉽습니다. 왜냐하면, 계속 변형이 일어나고 범위가 제한당하면서 경우에 따라 세밀하게 


표현 차이가 줄어드는 구조를 갖고 있기 때문입니다. 일단 $W_h$가 곱해지는 것은, 변형이 일어난다는 의미입니다. 그런 다음 tanh에 들어가면, 수치 범위가 -1과 1사이로 압축됩니다. 이렇게 되면, 절대값이 큰 숫자들은 그 차이가 매우 작아집니다. 그리고 이 변형과 범위 한정이 반복해서 일어납니다. 요컨대 $X_t$가 가졌던 정보는 지속적인 비선형변환을 거치게 되고 그 과정에서 원래의 형태와 특성을 잃게 될 가능성이 큽니다. 신호라는 관점에서 보면, 연산을 거듭할수록 앞에 들어온 신호의 특이성이 약해진다는 말이죠. 

그런데 제가 보기에 이걸 단순한 한계라고만 보기에는 약간 애매한 면이 있는거 같습니다. 구조적 특징 혹은 거기서 나온 자연스러운 부산물이라고 봐도 될만한 부분인 듯합니다. 즉 RNN은 '과거 방향으로 근거리에 있는 입력값들이 현재 시점에서 좀 더 강하게 작동하는' 구조적 특징을 가졌다고 할 수도 있다는 말이죠. 앞서 말했듯 RNN이 tanh를 주요 활성화함수로 삼고 있기에, 각 시점의 은닉 상태 값은 -1과 1사이로 한정됩니다. 덕분에 모든 시점의 RNN 블럭 출력이 동일한 범위 내에서 존재하면서 같은 용도로 활용될 수 있지요. 그리고 이에 따르는 자연스러운 부산물이 앞에서 들어온 값의 영향력이 여러 번의 변환을 거치면서 약해지기 쉽다는 점입니다. 가중치 행렬이 계속 곱해지는 것도 그런 효과를 내죠. 그래서 관점에 따라서는 '정보의 약화'를 RNN의 구조적 특징이라고 볼 수도 있을 듯합니다. 거꾸로 원거리 요소와 근거리 요소를 무차별하게 고려하면, 순서 정보를 제대로 살리지 못하는 일도 생각해 볼 수 있으니까요. 다만 우리가 RNN의 구조적 특징 혹은 편향을 넘어서서 언어에 존재하는 먼거리 의존을 포착하려니 이게 약점이 되는 경우가 많습니다. 

사실 제가 보기에 정보의 약화 문제보다 더 큰 문제는 **정보의 중첩 문제**입니다. 왜냐하면 RNN은 결국 매 시점마다 '고정 길이 벡터' (fixed length vector)로 정보를 축약하거든요. 위의 도식을 보면, 매 시점 정보가 hidden state로 압축될 때, 정해진 길이의 벡터로 한정됩니다. 그래서 RNN을 실제로 파이토치 등으로 구동해보면, 가장 중요한 하이퍼 파라미터가 hidden state 벡터의 차원입니다 (그러니까, hidden state가 몇 개의 숫자를 뭉쳐놓은 벡터냐가 중요하다는 말입니다). 즉 정해진 차원의 벡터에 앞에서 부터 계속 정보를 누적해서 기록하는 것입니다. 그러면, 필연적으로 '덮어써지는 부분'이 생깁니다. 

앞에서 $X_t$가 계속 변환된다는 말씀을 드렸습니다. 그런데 그보다 더 큰 문제는 $X_t$에서 기인한 정보가 새로운 시점의 입력값이 만들어내는 효과와 계속해서 섞인다는 점입니다. 즉 중첩됩니다. RNN 모델에 따르면 $X_t$이후에도 계속 입력이 들어오죠. $X_{t+1}$, $X_{t+2}$ 등이 들어옵니다. 그리고 거기서 기인한 수치가 계속해서 결합됩니다. 동일한 차원의 벡터 안에서 새 정보와 기존 정보가 반복적으로 섞이는 셈입니다.  

비유적으로 이미지로 표현해봅시다. 아래 같은 형국입니다. 

<div style="text-align:center;">
        <img src="_static/figures/rnn 정보 중첩 비유.jpg" width="700">
</div>

*이 그림은 조지아텍 Mark Riedl 선생님의 CS7650 Natural Language 강의 중 RNN 한계 설명 슬라이드에서 사용된 이미지를 바탕으로, 본 강의의 예시에 맞게 단순화하고 한글로 재구성한 것입니다. 이 장의 RNN과 LSTM 설명을 구성하는 과정에서도 해당 강의를 참고했습니다.*

자 이게 실제 연산 과정이라는 뜻은 아닙니다. 비유적으로 설명하는 것입니다. RNN은 고정된 차원의 벡터로 여러 시점의 입력 정보를 요약하고 있습니다. 그래서 모든 입력이 '한정된 칸'을 함께 써야 합니다. 그러다보니 중간에 지워지고 덮어써지는 면이 존재할 수 밖에 없지요. 특히 처음에 기록된 것들이 그럴 것입니다. 정보의 약화는 이론적으로 보면 가중치 행렬의 원소들이 잘 학습되면 극복될 가능성이 없지는 않습니다. 하지만 이런 정보의 겹침은 사실 해소되기가 힘듭니다. 

역전파 상황에서도 문제가 관찰됩니다. 특히 정보의 약화 문제가 그렇습니다. 아예 따로 이름도 붙어 있습니다. **그레디언트 소실과 폭발**이라는 이름이죠. 특히, 언어에서 토큰 간 장기 의존이 존재하고, 그래서 멀리서 발생한 손실 정보가 긴 거리를 이동한 후 파라미터 수정 압력으로 작동해야 하는 경우 이게 문제가 됩니다. 

앞선 개요 강의에서 언급했듯이 오차 역전파의 핵심은 '손실'을 최소화하는 방향으로 파라미터를 조정하는 것입니다. 그럼 다시 앞선 예를 살펴보죠. '원광이는 자기 방 컴퓨터 의자에 앉아서 작업에 열중하고 있었다. 혜미가 들어와 ??에게 턱짓으로 인사했다.' 여기서 우리 모델이 ??을 '원광이'로 제대로 예측하지 못하고 '의자'라고 예측했다고 하죠. '원광'은 후보군에 있었지만 확률을 0.01% 정도로 부여해서 손실이 매우 컸다고 합시다. 우리의 모델이 잘 학습되려면, 이 손실이 주는 신호가 '원광이'가 입력되는 위치까지 잘 전달되어서 '원광이'라는 토큰이 입력될 때 그 정보를 향후에 고려할 수 있도록 관련 파라미터를 조정해야 합니다. 

그런데 그게 지금 연산 구도상 쉽지 않습니다. 역전파가 일어나면, 특정 시점의 손실이 은닉 상태에 대해 가지는 그레디언트가 어떤 과정을 겪으면서 전달될까요? 일단 tanh의 도함수가 곱해질 것입니다. 그런데 tanh의 도함수는 0과 1 사이의 값을 가집니다. 그래서 사실 그레디언트에 0과 1 사이의 값이 계속 곱해지는 (원소별 곱 형태로) 꼴이 됩니다. $W_h$도 계속 곱해집니다. 우리가 앞서 오차 역전파에서 배운 것을 RNN 도식에 적용한다고 생각해보세요. $H_t = tanh(W_x \cdot X_t + W_h \cdot H_{t-1} + b)$ 이렇게 생긴 핵심 연산에 위에서 미분 계수가 내려오고, 결국 $H_{t-1}$에 대해 가지는 그레디언트가 다시 뒤로 전달될텐데, 그러면 일단 tanh 도함수가 내려온 그레디언트에 곱해지고, 그 다음에 $W_h$ 곱해집니다. 식으로 쓰면 아래와 같습니다. 

$$
\begin{aligned}
H_t &= \tanh(W_xX_t + W_hH_{t-1} + b) \\[10pt]
a_t &= W_xX_t + W_hH_{t-1} + b \\[10pt]
\frac{\partial L}{\partial H_{t-1}}
&=
W_h^\top
\left(
\frac{\partial L}{\partial H_t}
\odot
\tanh'(a_t)
\right)
\end{aligned}
$$


10강에서 오차 역전파 강의를 잘 따라오셨다면 이렇게 나온 이유를 이해하실 겁니다. 이 그레디언트 전달 식이 의미하는 바는, 위에서 내려온 손실이 H에 대해 가지는 그레디언트에 0과 1사이의 값이 계속 곱해지고, 동시에 같은 행렬이 계속 곱해진다는 것입니다. 그러면 많은 경우 수치가 0을 향해 가버리는 문제가 생깁니다. 즉 각 시점의 은닉상태가 손실에 대해 가지는 그레디언트가 0을 향해 간다는 말입니다. 그러면 여기에 각 시점의 입력값이나 이전 은닉 상태를 곱해서 해당 시점의 가중치 행렬($W_x, W_h$)의 그레디언트를 구하는데, 그것도 매우 작아지겠죠. 이렇게 그레디언트가 0에 가까우면 무슨 일이 생길까요? 학습이 제대로 이루어지지 않습니다. 경사하강법 자체가 그레디언트를 활용하고 있으니까요. 그래서 이런 현상을 그레디언트가 사라지는 문제라는 의미를 담아 **'그레디언트 소실 문제'** 라고 부릅니다. '정보의 약화' 문제의 역전파 버전인 셈이죠. 

물론 원칙적으로 $W_h$를 구성하는 수치의 절대값이 매우 크면, 거꾸로 그레디언트가 너무 커지는 문제가 생길 수도 있습니다. 이걸 **'그레디언트 폭발 문제'** 라고 합니다. 하지만 '정보'라는 관점에서는 효과가 비슷합니다. 그레디언트가 너무 커지면 반대로 0에 가까울 때와 마찬가지로 유용한 학습 신호로 기능하기 어렵습니다. 그레디언트가 너무 커진다고 생각해봅시다. 그러면 학습이 극단적인 방향으로 일어납니다. 그래서는 적절한 최적점을 찾아 파라미터를 섬세하게 변동하기 어렵습니다. 즉 어떤 파라미터의 그레디언트가 폭발해 버리면, 학습 방향이 극단적이 되고, 그래서 모델 성능이 좋을 수가 없습니다.

요컨대 역전파 상황에서도 정보의 약화 문제가 생깁니다. 그레디언트라는 또다른 중요한 정보가 먼 거리를 이동하면서 0으로 가버리거나 폭발해버리는 일이 연산 구조상 생기기 쉽습니다. 그리고 이건 먼 거리 의존을 고려한 작동을 학습하기 어려운 구조적 성질이 되죠. 

## 4. RNN 한계의 극복 노력

### 4.1. LSTM

#### 4.1.1. LSTM의 설계와 두 가지 목표 과제

RNN의 패러다임 안에서 이런 한계를 극복하려는 노력은 많았습니다. 이 강의에서는 핵심적인 두 가지 아이디어를 살펴보겠습니다. 하나는 LSTM이고, 다른 하나는 Attention입니다. 

우선 LSTM은 정보의 약화와 겹침 문제를 다음과 같은 조치를 통해 완화하려 합니다. 

1) 정보 흐름을 cell state와 hidden state로 복수화하여 나누고 이 복수의 정보 흐름을 통제하는 여러 '게이트'(gate)를 도입합니다. 이를 통해 정보의 중첩 문제를 완화하려 시도합니다. 
2) cell state 갱신 과정에서 정보의 약화 문제를 완화할 수 있는 구조를 도입합니다. 

이를 이해하기 위해 우선 LSTM의 아이디어를 간략히 살펴보겠습니다. 앞서 말씀드렸듯 LSTM의 핵심 조치 중 하나는 맥락 정보를 둘로 나누는 것입니다. cell state와 hidden state가 그것이죠. 그리고 이 맥락 정보들을 산출하는데, '게이트'라는 것을 활용합니다. 그러니까, 우리가 앞에서 봤던 RNN 연쇄 모델을 LSTM으로 바꾸면 아래와 같이 됩니다. cell state는 C로, hidden state는 H로 표기하고 있습니다. 

<div style="text-align:center;">
    <img src="_static/figures/LSTM 기본 도식.jpg" width="500">
</div>

즉 이 그림처럼, 원래는 H를 시점마다 생성하고 그걸 다음 시점 연산에 개입시키는 것이었는데, 이제는 C와 H라는걸 만들고 그걸 다음 시점 연산에 개입시킵니다. 그러면 구체적으로 C와 H는 어떻게 산출되는 것일까요? 그리고 게이트라는건 어떻게 작동하는걸까요? 이를 위해서는 그림에 LSTM이라고 되어 있는 블럭 내부의 연산 구조를 봐야 합니다. 그 구조를 그림으로 표현하자면 다음과 같습니다. 


<div style="text-align:center;">
    <img src="_static/figures/LSTM 블럭.jpg" width="800">
</div>

    이 그림은 조원광이 Dive into deep learning의 Fig 10.1.3을 주요 참고 자료로 삼아 PPT로 그렸습니다. 

자 RNN보다 훨씬 복잡해 보이죠? 이게 LSTM 블럭입니다. RNN 블럭이 있던 자리에 이 LSTM 블럭이 들어가는 셈입니다. 즉 원래 RNN에서는 $H_t = tanh(W_x \cdot X_t + W_h \cdot H_{t-1} + b)$ 이 연산만으로 다음으로 넘길 은닉 상태를 만들고, 거기에 추가작업을 해서 각 시점별 출력을 만들었는데, LSTM은 이게 훨씬 복잡해집니다. 일단, 전 블럭에서 넘어오는 정보가 두 가지가 되었고 ($C_{t-1}, H_{t-1}$), 다음으로 넘어가는 것도 두 가지입니다 ($C_{t}, H_{t}$). 그리고 보통 t 시점 결과인 $H_{t}$에 추가 작업을 해서 t 시점의 핵심 출력값을 만들곤 합니다. 

왜 이렇게 복잡하게 하는 것일까요? 간단히 설명하자면, **'각 시점 별로 추가로 기억해서 멀리 전달할 것과 이제는 잊어도 될 것을 판단하여 보다 섬세하게 맥락 정보를 만들기 위해서'** 라고 할 수 있습니다. 

앞서 말씀드렸듯이 RNN은 정보의 중첩이라는 문제를 가졌습니다. 이걸 해결하기 위해 LSTM은 맥락 정보를 복수화합니다. C라는걸 도입하죠. 그리고 역할을 부여합니다. C는 '오랫동안 유지할 정보'를 보관하는 벡터용으로 설계되었습니다. 즉 맥락 정보를 H라는 하나의 벡터에 담는게 아니라, H와 별도로 C를 도입하여 상대적으로 원거리를 거쳐 중요 정보를 전달하는 기능을 담기로 했다는 말입니다. 

다만 C를 그냥 도입만 해두면, C도 H도 앞서 살펴본 문제, 특히 중첩 문제에서 자유로울 수 없을 것입니다. 특히 C는 상대적으로 원거리 전달용으로 설계되었으니, 중첩을 최대한 완화해야 합니다. 그러지 않으면 원거리 전달이 불가능하죠. LSTM은 일단 중첩을 극복하기 위해 '게이트' 라는 걸 도입합니다. 게이트는 정보의 흐름에서 '추가로 기억해야 할 것, 이제는 잊어야 할 것, 이 시점의 산물로 삼아야 할 것'을 선택하여 전체 정보 흐름을 조절하는 장치입니다. LSTM은 이걸 세 개 운용합니다. Forget gate (뭘 잊어야 하는지 알려주는 것), Input gate (추가로 기억해야 할 것을 알려주는 것), Output gate (이렇게 만들어진 C에서 이 시점의 산물인 H로 남길 것을 알려주는 것) 이 그것입니다. 

이런 여러 조치가 합쳐져서 모델이 오랫동안 유지해야 할 정보를 식별하여 유지할 수 있게 됩니다. 그냥 무턱대로 계속 섞어 버리는게 아니라, 길게 가져갈 것은 식별해서 유지하고, 지울건 지우는 장치를 만든 셈입니다. 이를 통해 RNN이 보인 정보의 중첩이라는 문제를 어느 정도 완화하려는 것이죠

하나하나 간단히 살펴봅시다. 다시 한 번 요약하자면, LSTM의 핵심 취지는 <C를 도입하고, 그것을 상대적으로 장기 정보를 안정적으로 전달하는 용도로 쓰고, 이것에 기록할 것과 지워야 할 것을 유연하게 조절한다>는 것이었습니다. 

일단 과거로부터 C와 H가 넘어온 상황에서 출발하겠습니다. 즉 $C_{t-1}, H_{t-1}$이 있습니다. 우리가 해야 할 일은 두 가지입니다. **첫 번째, 지금 t 시점에서 $C_{t-1}$에서 지워야 할 것을 판단하고, 이번 시점에서 새로 기록해야 할 것을 마련합니다. 이것에 근거하여 새롭게 $C_t$를 만듭니다. 두 번째, 이렇게 만들어진 $C_t$까지 고려해서 지금 시점의 핵심 결과, 즉 $H_t$를 만듭니다.** 첫 번째 과제는 forget gate와 input gate를 동원한 여러 조치를 통해 이루어지고, 두 번째 과제는 output gate를 동원한 조치를 통해 이루어집니다. 

#### 4.1.2. 첫 번째 과제: $C_t$ 만들기

**Forget gate**

우선 첫 번째 과제부터 봅시다. 작업은 간단합니다. 과거에서 넘어온 $C_{t-1}$에서 지울걸 지우고, 더할걸 더합니다. 먼저, 지우는 파트는 forget gate가 핵심적인 역할을 합니다. forget gate라고 쓰니 무슨 물리적인 장치를 만드는 거 같은데, 그게 아니라 그냥 그런 역할을 하는 벡터를 만드는 것입니다. 그리고 원래 $C_{t-1}$에 이 게이트에 해당하는 벡터를 원소별 곱을 해줘서 마치 물리적 게이트와 같은 효과를 내는 것입니다. 위에 제가 그린 LSTM 블럭 그림의 맨 왼편을 보시죠. forget gate가 되는 벡터는 아래와 같은 수식으로 만들어집니다. 

$$
\begin{aligned}
f_t = sigmoid(W_{xf}X_t + W_{hf}H_{t-1}+b_f)
\end{aligned}
$$

자 이 수식의 의미를 생각해보죠. 일단 $W_{xf}, W_{hf}, b_f$는 t 시점의 forget gate를 만들기 위한 forget gate 전용 가중치 행렬과 편향입니다. 즉 지금 시점의 입력과 전 시점의 은닉 상태를 선형 변환하여 지금 시점에서 forget gate로 기능할 벡터의 원형태를 만들어내는 것입니다. 이건, forget gate가 결국 '지금 시점에 판단해 보니 이건 지우는게 좋겠어'라는 판단을 해줘야 하니 자연스럽습니다. 지금 시점의 입력과 전 시점의 은닉 상태를 종합해서 이 판단을 내려야 하니까요. 그런 다음 그게 sigmoid에 들어갑니다. 그러면 어떻게 될까요? 벡터를 구성하는 모든 값이 0과 1사이로 압축됩니다.  

그래봐야 이게 벡터인데, 어떻게 게이트로서의 역할을 하는가 싶으실 겁니다. 그런데 만약 이걸 기존의 $C_{t-1}$에 원소별 곱 형태로 곱해주면 어떨까요? 예를 들어 기존 $C_{t-1}$가 [2,1,4] 였는데, 거기에 가상의 forget gate가 [0.5, 0.00001, 0.99999]가 원소별 곱이 된다고 해볼게요. 그러면 결과는 [1,0.00001,3.99996] 같은 결과가 나옵니다. 이게 뭘 의미하는 것일까요? 이건, $C_{t-1}$에서 세 번째 값은 사실상 그대로 유지되고 두 번째 값은 사실상 무화됩니다. 잊어버리는 셈이죠. 그리고 첫 번째 값은 반만 살아서 전달되네요. 

요컨대, 시그모이드를 통과하여 0과 1사이의 값을 구성요소로 하는 벡터는, 같은 차원의 벡터에 원소별 곱으로 곱해지면, 일종의 '연속적 스위치 혹은 게이트' 같은 기능을 합니다. 1에 가까운 값을 곱하는 건, '그대로 기억해라'라고 말하는 셈이 되고, 0에 가까운 값을 곱하는건 '잊어라'라고 말하는 셈이되는 거죠. 그 중간도 존재할 수 있고요. 그래서 위의 절차를 거쳐 만들어진 벡터를, 숫자임에도 'forget gate'라고 말하는 것입니다. 

여기서 중요한 것은, 이 forget gate의 설정이 '학습'에 의해 정해진다는 것입니다. $W_{xf}, W_{hf}, b_f$가 사실상 이 스위치의 작동 방식을 결정한다고 볼 수 있습니다. 그런데 이들은, 사용자가 지정하는게 아니라 데이터로부터 학습됩니다. 그러니 forget gate는, 일종의 '학습 가능한 연속적인 스위치/게이트'가 되는 것이지요. 여담이지만, 이런 방식으로 '미분 가능하여 경사하강법으로 학습 가능한 연속적 스위치 혹은 게이트'를 구현하는 일은 LSTM이외에도 제법 많습니다. 

**Input gate**

자 첫 번째 과제를 마무리하기 위해서는 지운 이후 더할걸 더하는 일이 필요합니다. 계속 지우기만해서는 장기 전달 정보가 유용해지지 않겠지요. 더하는 파트는 input gate가 개입합니다. 지우는 것보다는 약간 구조가 복잡합니다. 제가 그려둔 그림에서 forget gate 옆의 두 개의 구조를 보시죠. 일단 input gate를 만듭니다. 그건 forget gate와 완전히 동일한 방식으로 만듭니다. 다만 input gate용 파라미터들을 쓰는거죠. 즉 t 시점에 어떤 정보를 얼마만큼 남겨서 더할지 결정하는 스위치를 아래와 같이 만듭니다. 

$$
\begin{aligned}
i_t = sigmoid(W_{xi}X_t + W_{hi}H_{t-1}+b_i)
\end{aligned}
$$

그런데 이건 스위치/게이트입니다. 즉 뭔가 주어지면 그것 중에 뭘 얼마나 살리고 죽일지 (너무 살벌하다면^^;; 비활성화할지) 결정하는거죠. forget gate의 경우, 앞에서 넘어온 $C_{t-1}$이 있었으니 거기에 그냥 원소별 곱을 하는 방식으로 작동하면 됩니다. 그런데 input 같은 경우는 지금 스위치/게이트는 만들었는데 내용이 아직 없습니다. 그래서 '더해야 할 내용' 자체를 만들어 내야 합니다. 그걸 지금 세 번째 Candidate cell state가 하고 있습니다. 이건 아래와 같이 계산됩니다. 

$$
\begin{aligned}
\tilde{C}_t = tanh(W_{xc}X_t + W_{hc}H_{t-1}+b_c)
\end{aligned}
$$

앞에 두 게이트와 다른 점은 활성화 함수 뿐입니다. Candidate cell state를 tanh를 거쳐 만들어내는 것은 자연스럽습니다. 왜냐하면 이건 '내용' 이거든요. 숫자가 표현할 수 있는 내용 중 '부호'는 핵심적 요소입니다. 그러니 -1부터 1까지의 출력 범위를 갖는 tanh가 sigmoid보다 적절하죠. 그리고 이건 일정 범주 안으로 수치를 한정하는 효과도 동시에 가집니다. 그러지 않으면 cell state에 추가로 들어가는 수치의 범위가 시점 마다 너무 불균형할 수도 있습니다. 

그런 다음, 이 candidate cell state로 마련된 벡터에 input gate로 마련된 벡터를 원소별 곱을 합니다. 그러면, '후보로 마련된 내용' 중 일부는 꽤 통과될 것이고 일부는 꽤 비활성화되겠죠. 그게 input gate가 하는 일이니까요. 그리고 그걸 $C_{t-1}$에서 forget gate를 통과한 값에 합쳐주는 거죠. 이게 새로운 cell state, 즉 $C_t$가 됩니다. 


**첫 번째 과제 정리**

즉 첫 번째 과제인 **<지금 t 시점에서 $C_{t-1}$에서 지워야 할 것을 판단하고, 이번 시점에서 새로 기록해야 할 것을 마련합니다. 이것에 근거하여 새롭게 $C_t$를 만든다>** 는 수식적으로 표현하면 아래와 같이 진행됩니다. 

$$
\begin{aligned}
C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t
\end{aligned}
$$

$\odot$ 표시는 원소별 곱을 표현하는 것입니다. 벡터나 행렬에 '곱'을 하는 방법이 여럿이라 이렇게 특별히 표현합니다. 

즉 이건, 새로운 Cell state는 기존의 값에서 forget gate를 적용하여 잊을 걸 잊고 (정확히 말하면 연속적인 비활성화를 통과하고) 지금 시점에서 후보값을 마련한 후 그것에 input gate를 적용해서 더하는 방식으로 이루어집니다. 

저는 처음에는 forget gate까지는 쉽게 납득이 되었습니다. 그런데 후보값을 따로 마련해서 input gate를 걸어서 결과를 만든 후 그걸 더해주는 과정은 약간 의아했습니다. 그냥 후보값만 더해줘도 되지 않나 싶었거든요. 어차피 후보값을 만들어내는 파라미터들 ($W_{xc}, W_{hc}, b_c$)이 적절히 학습된다면, input gate를 따로 걸지 않아도 적절한 숫자로 생성될 수 있을테니까요. 

그런 방식도 원리적으로 불가능하지는 않을 것 같습니다. 아마도 LSTM은 여기서 작업을 개념적으로 둘로 구분하고, 학습 가능성을 더 높이려 하지 않았나 싶습니다. 'C에 더할 새로운 값을 만들어낸다'는 것은 크게 두 가지의 작업을 포함합니다. 첫 번째는 방향이 포함된 내용을 만들어 내는 것. 두 번째는 그 내용의 강도를 조절하는 것입니다. Candidate cell state를 만들어내는 것이 첫 번째를 담당합니다. Input gate가 두 번째를 담당합니다. 물론 Candidate cell state를 만드는 파라미터가 '내용'과 '강도'를 함께 표현하도록 훈련될 수도 있으나, 이렇게 둘로 구분하면 좀 더 자유도가 높은 상태에서 이 두 개의 작업을 원활히 할 수 있습니다. 아마도 이런 의도였지 않았나 싶습니다. 

#### 4.1.3. 두 번째 과제: $H_t$ 만들기

**Output gate**

자 그럼 두 번째 과제를 생각해봅시다. 두 번째 과제는 $H_t$를 만드는 것입니다. $C_t$가 다음으로, 더 나아가 오랫 동안 전달된 정보를 위한 것이라면, $H_t$는 지금 시점의 핵심 결과물이자 바로 다음 시점의 연산에 영향을 미치게 만드는 정보입니다.

이걸 만드는데, 우리가 앞에서 만든 $C_t$를 개입시켜야 합니다. C를 장거리 정보 전달용이라고 말했는데, 정말 전달만 하면 쓸모가 없습니다. 오래 유지되는 정보를 갖고, 그때그때 적절히 결과 산출에 기여하도록 해야 합니다. 그렇기에, $H_t$는 일단 지금 시점의 입력인 $X_t$ 그리고 전 시점에 넘어온 은닉 상태인 $H_{t-1}$ 그리고 우리가 앞에서 살펴본 과정을 통해 만들어낸 $C_t$를 종합해서 만들어야 합니다. 

LSTM이 이걸 해내는 방식은 아래 수식과 같습니다. 위에서 제가 제시한 그림의 마지막 파트를 보시면 쉽습니다. 

$$
\begin{aligned}
o_t &= sigmoid(W_{xo}X_t + W_{ho}H_{t-1}+b_o) \\[15pt]
H_t &= o_t \odot tanh(C_t)
\end{aligned}
$$




일단 $H_t$의 내용은 $C_t$에서 옵니다. 거기에 tanh 활성화함수를 걸어서 기본 내용을 준비합니다. tanh를 거는 이유는, 비선형 변환의 이유와 값의 범주를 일정하게 (-1과 1 사이) 유지하려는 목적으로 해석할 수 있습니다. 새로 $C_t$를 만드는 과정에서 이번 시점에 기록할 값이 더해졌기에 값들의 분산이 커져있는 상태니까요. 

그런 다음, 이 내용 중 뭘 얼마나 살려서 $H_t$로 삼을지 준비하는 output gate를 앞서 두 개의 게이트와 같은 방식으로 만듭니다. 그리고 그걸 원소별 곱을 해주는 거죠. 이 역시 마찬가지로 꼭 게이트/스위치가 필요한 상황은 아닐 수 있습니다만, '내용 준비'와 '강도 조절'을 별개의 파라미터 셋으로 해내려고 하는 것이라 해석할 수 있습니다. 

물론, 반드시 이런 방식으로 $H_t$ 만들어야 할 필연적 이유가 있다고 생각하지는 않습니다. 세 정보 ($C_t, X_t, H_{t-1}$)을 종합하는 방법은 원리적으로 다양하죠. 다만 해석하자면, LSTM은 C를 새로 도입한 만큼, 그것을 중심으로 H를 만드는 전략을 택한 듯합니다. 그래야 '장기 기억을 유연하게 조성하여 결과에 영향을 미치도록 한다'는 취지가 살지요. 하지만 이게 C가 H에 비해 압도적으로 주도권을 갖는다고 해석하기도 어려운 것이, C의 갱신에 H가 영향을 미치고 있습니다. 정확히는 전 시점의 H가 영향을 미치죠. 즉 $C_t$는 $X_t$와 $H_{t-1}$을 재료로 갱신되고, 다시 $C_t$를 베이스로 $H_t$를 만들고 있는 셈입니다. 이건 상대적 독립성을 유지하되 서로 영향을 미칠 수 있도록 되어 있는 구조라 할 수 있습니다. 그리고 실제로 성공적이었습니다. 

#### 4.1.4 기여와 한계

자 그러면 이제까지 나온 설명을 종합해서, LSTM 블럭에서 일어나는 일을 수식으로 종합하면 이렇습니다. 위에 나온 그림은, 이걸 정보 흐름에 맞게 표현한 것이지요. 

$$
\begin{aligned}
f_t &= sigmoid(W_{xf}X_t + W_{hf}H_{t-1}+b_f) \\[15pt]
i_t &= sigmoid(W_{xi}X_t + W_{hi}H_{t-1}+b_i) \\[15pt]
o_t &= sigmoid(W_{xo}X_t + W_{ho}H_{t-1}+b_o) \\[15pt]
\tilde{C}_t &= tanh(W_{xc}X_t + W_{hc}H_{t-1}+b_c) \\[15pt]
C_t &= f_t \odot C_{t-1} + i_t \odot \tilde{C}_t \\[15pt]
H_t &= o_t \odot tanh(C_t)
\end{aligned}
$$


바닐라 RNN보다는 꽤 복잡해졌죠? 그리고 이렇게 하는 취지는 오래 지속시키면서 활용할 만한 정보를 유연하게 마련하여 시퀀스를 따라 전달하고 (C), 그 정보까지 활용하여 매 시점 은닉 상태를 순차적으로 만들기 위함 (H) 이었습니다. 이런 디자인에서 데이터에 따라 모델이 잘 학습되면, 바닐라 RNN에서처럼 의도치 않게 정보가 중첩되면서 무화되는 일이 완화될 수 있습니다. 실제로 LSTM은 여러 작업에서 꽤 괜찮은 성능 개선을 가져왔습니다. 

지금까지 설명은 사실 순전파 상황에 초점을 둔 설명이었습니다. 하지만 LSTM의 장점은 역전파 상황에서도 나타납니다. C의 갱신 경로는 그레디언트 정보가 쉽게 소실되지 않고 초기 블럭까지 잘 전달되는데 도움을 줍니다. 역전파 상황에서 정보의 약화 문제를 완화하는 거죠. 위의 정리해서 봤듯이, C의 갱신 식은 이렇습니다. $C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$ 그레디언트 정보가 잘 전달된다는 말은, $\frac{\partial{L}}{\partial{C_t}}$ 라는 그레디언트가 내려왔을 때, 그게 $\frac{\partial{L}}{\partial{C_{t-1}}}$이 되면서 구조적 특징 때문에 소실되거나 증폭되는 일이 적다는 말입니다. 그러면 LSTM에서 $\frac{\partial{L}}{\partial{C_{t-1}}}$은 어떻게 계산될까요? 직접 경로만 보자면 아래와 같이 계산됩니다. 

$$
\begin{aligned}
\frac{\partial{L}}{\partial{C_{t-1}}} = \frac{\partial{L}}{\partial{C_t}} \odot f_t
\end{aligned}
$$

$C_t$가 갱신된 식이 이랬습니다. $C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$ 이걸 되집어 가는겁니다. 앞서 10장에서 배웠듯이 덧셈이 계산 그래프에 있을 경우 그레디언트는 각 항목에 그대로 배분됩니다. 즉 딱히 영향을 미치지 않죠. 그런 다음 $C_{t-1}$의 그레디언트가 갱신되어야 하는데, 그건 위에서 내려온 그레디언트에 $C_{t-1}$에 곱해졌던 $f_t$를 곱해서 구합니다. 

엄밀히 보면, $C_{t-1}$의 그레디언트는 이것 말고 H를 경유한 신호에도 영향을 받습니다. 왜냐하면 $C_{t-1}$이 $C_t$가 되는 과정에서 작동하는 숨은 과정이 있기 때문이죠. $C_{t-1}$은 직전 블록에서 $H_{t-1}$을 만드는데 기여했습니다. 그리고 $H_{t-1}$은 $f_t, i_t, o_t, \tilde{C}_t$의 형성에 영향을 미치죠. 그리고 이것은 $C_t$의 조성에 이어집니다. 그러니, $C_t$의 그레디언트도, 직접적으로 $C_{t-1}$의 그레디언트로 가는 부분도 있지만, 이 간접 경로를 거쳐서 $C_{t-1}$의 그레디언트로 가는 부분도 있습니다. 

다만 이 직접 경로의 존재가 큰 차이를 만듭니다. 이 직접 경로를 중심으로 우리가 봤던 바닐라 RNN과 사뭇 다릅니다. 바닐라 RNN에서는 C가 없었고 H를 통해 정보가 시퀀스를 타고 전달되었습니다. 그때 H의 변형식과 그레디언트의 역전파 식을 다시 살펴보면 아래와 같습니다. 

$$
\begin{aligned}
H_t &= \tanh(W_xX_t + W_hH_{t-1} + b) \\[10pt]
a_t &= W_xX_t + W_hH_{t-1} + b \\[10pt]
\frac{\partial L}{\partial H_{t-1}}
&=
W_h^\top
\left(
\frac{\partial L}{\partial H_t}
\odot
\tanh'(a_t)
\right)
\end{aligned}
$$


LSTM에서 C에 대한 그레디언트가 전달되는 직접 경로와 바닐라 RNN에서 H에 대한 그레디언트가 전달되는 경로를 비교해보죠. 가장 큰 차이는 tanh의 도함수가 곱해지는 계산이 존재하는 점입니다. 그리고 10장에서 봤듯이 tanh의 도함수는 0과 1 사이의 범위를 갖습니다. 앞서 말했듯 이게 계속 곱해지면, 아무래도 값이 커지기 힘들죠. 그레디언트가 소실되는 좋은 배경이 된다는 말입니다. 그런데 LSTM에서 C와 관련된 주 경로에는 그 부분이 없습니다. 이건, C를 통해서 그레디언트가 역전파되면서 상대적으로 적게 소실될 수 있는 중요한 배경입니다. 물론 $f_t$즉 forget gate가 계속 곱해지면서 그레디언트가 소실될 수 있습니다. 예를 들어 특정 위치의 f값이 계속 0에 가깝게 나오면, 거기의 정보는 소실되겠죠. 하지만 거꾸로 이는 f_t의 각 값들이 1에 가깝게 학습되어 있다면 (즉 순전파 상황에서 특정 부분을 통과시키게 되어 있다면) 그레디언트도 그냥 통과될 수 있다는 말입니다. 즉 forget gate는 그레디언트 정보를 조절하는 게이트이기도 한거죠. 

바닐라 RNN에서도 $W_h$가 특정 부분에서 그레디언트가 소실되거나 증폭되지 않도록 해당 값을 1에 가깝게 학습하면 되지 않는가? 라는 의문이 생길 수 있습니다. 저도 원리적으로 그럴 수 있다고 생각합니다. 하지만 이게 LSTM보다 훨씬 어렵습니다. 왜냐하면 $W_h$가 해야 하는 일이 순전파와 역전파 상황에서 '어떤 정보를 얼마나 살려서 전달할 것인가?'에 대답하는 일만 있는게 아니거든요. 바닐라 RNN 상황에서 $W_h$는 기존 은닉 상태를 어떻게 변형하여 지금의 은닉 상태를 창출할 때 써야 할지도 담당하고 있습니다. 보존할거냐 무화할거냐만 담당하는게 아니라는 것입니다. 그런데 LSTM에서는 여러 기능들이 별개의 파라미터로 분리되어 있습니다. 잊을거냐 남길거냐에 해당하는 파트는 forget gate 파트가 담당하고, 뭘 새로운 C로 삼을 것인지는 input gate와 candidate cell state 파트가, 뭘 새로운 H로 삼을지는 output gate가 담당하고 있죠. 그러니, forget gate는 상대적으로 특정 시점에서 어떤 정보들을 얼마나 보존할지 집중해 학습될 수 있는 구조입니다. 

요컨대 LSTM은 순전파 상황에서 정보의 중첩 문제를 완화합니다. 앞서 설명한 설계 덕분입니다. 역전파 상황에서도 그레디언트의 소실과 폭발 문제를 완화합니다. 직접 경로만 보면, 곱해지는 $f_t$가 항상 0과 1사이의 값을 가지기에, 폭발 문제는 일어나기 힘듭니다. 적어도 직접 경로에서는 그렇다는 말입니다. 소실 문제는, 원리상 일어날 수 있지만, tanh의 도함수를 통과하지 않는다는 점과 $f_t$의 적합한 학습이 바닐라 RNN보다 훨씬 용이하기에 상대적으로 안정적입니다.

### 4.2. Attention

하지만 LSTM이 바닐라 RNN이 가졌던 문제를 완전히 해결하고 있다고 보기는 어렵습니다. 특히 정보의 중첩 문제가 그렇습니다. Cell state라는 추가 장치를 도입하여 거기에 기록될 정보를 유연하게 조절하면서 필요할 경우 오래 유지하려고 했지만, 어쨌든 정보를 '고정 길이 벡터'에 한정하여 담으려고 한다는 점은 바닐라 RNN과 다르지 않습니다. 그 역할을 하는 벡터를 둘로 나누고 서로 다른 역할을 하도록 유도하는 구조를 지녔지만 (상대적으로 장거리 전달을 위해 마련한 cell state와 각 시점 출력과 다음 계산에 쓰이는 hidden state), 본질적으로 이 점은 변하지 않았죠. 

그래서 시퀀스가 길어지고 탐색해야 할 원거리 의존이 많아질 수록, LSTM이 시도한 개선 방식은 점차 한계를 드러낼 수 있습니다. 물론 '뭘 기억하고 뭘 지울지' 통제하는 게이트 시스템이 있지만, 그렇다고 '정보를 기록할 공간 자체가 부족한 문제'가 본질적으로 해결된 것은 아니니까요. 유연하게 기억할 것을 결정하면서 필요없는 것은 지우는 시스템을 마련했지만, 시퀀스 길이가 계속계속 늘어나면 이런 방법도 한계에 부딪히기 마련입니다. 아무리 저글링을 잘해도, 공 개수가 늘어나면 점점 한계에 부딪히기 마련인 것처럼요. 즉 LSTM은 정보의 겹침과 그에 따른 약화 문제를 완화하는 아이디어이기는 하지만, 고정 길이 벡터 때문에 생기는 근본적인 한계는 여전하다는 말입니다. 

**이 문제를 아예 다른 발상으로 극복하려 한 것이 바로 attention 메커니즘이었습니다. 이건 고정 길이 벡터 안에서 좀 더 결과를 정교하게 만드는 장치가 아니라, 아예 고정 길이 벡터로부터 오는 제약 자체를 벗어나려는 시도였습니다.** 

이걸 이해하기 위해 우선 어텐션에 대해 간단히 알아보겠습니다. 다른 방법도 그렇지만 어텐션이라는 장치도 여러 방식으로 구현될 수 있는데, 여기서는 Bahdanau attention으로 알려진 초기 어텐션 장치를 참조하여 설명하겠습니다. (이 아이디어는 Bahdanau, D., Cho, K., & Bengio, Y. (2014). Neural machine translation by jointly learning to align and translate. arXiv preprint arXiv:1409.0473. 에서 제안되었습니다. 유명한 한국인 NLP 연구자인 조경현 교수님도 참여한 기념비적 연구지요. 독자분들이 직접 읽어보는 것을 권합니다.)

우선 어텐션은 RNN을 가지고 하는 seq2seq 모델링의 성능을 개선하기 위해 처음 출현했습니다. 그리고 그것은 무척 성공적이었습니다. 나아가 이 아이디어는 seq2seq의 개선에만 머무르지 않고, RNN의 근본적 난점을 해결하고 Transformer라는 새로운 아키텍쳐를 도출하는 핵심적인 씨앗이 되었습니다. 지금부터 그걸 따라가 보겠습니다. 

이를 위해 seq2seq, 즉 sequence to sequence 모델링에 대해 좀 설명을 드려야겠습니다. 이게 뭐냐면, 입력이 시퀀스인데 출력이 시퀀스인 문제를 모델링하는 방법을 의미합니다. 이런 문제에서 입력 측 시퀀스와 출력 측 시퀀스는 길이가 다를 수 있습니다. 그리고 입력 시퀀스와 출력 시퀀스의 구성요소들이 정확히 순서대로 1:1로 대응되지 않습니다. 그래서 단순 시퀀스 레이블링과는 다릅니다. 

예를 들어 기계 번역을 생각해보죠. 한국어로 '어제 아버지와 함께 축구를 했다'라는 문장을 영어로 번역하면 'I played soccer with dad yesterday' 정도가 될거 같네요. 한국어 시퀀스가 들어가면, 영어 시퀀스가 나와야 합니다. 보다시피 길이가 다릅니다. 언어가 다르니 당연하죠. 1:1 대응도 애매한 경우가 생깁니다. 예를 들어 뒤쪽의 I에 정확히 대응되는 한국어 토큰은 애매하죠. 챗봇 같은 것도 마찬가지입니다. '서비스 센터 위치가 궁금해요'라고 입력이 들어가면 '서비스 센터 위치는 종각역 2번 출구입니다'라고 출력이 나와야 하죠. 길이가 다를 뿐만 아니라 입력 시퀀스와 출력 시퀀스의 1:1 대응도 애매합니다. 이런 모델을 seq2seq 모델이라 하고, 이걸 학습시키고자 하는 노력이 많았습니다. 많은 언어 활동을 이런 구도로 볼 수 있거든요. 대화가 사실상 seq2seq이니까요. 

보통 이걸 RNN을 활용해서 하곤 했습니다. 생각해보면 적합한 면이 많습니다. seq2seq은 근본적으로 입력이 시퀀스인데 거기에 있는 정보를 모두 종합적으로 고려하여 새로운 시퀀스를 생성해야 하는 상황을 의미하기 때문입니다. 보통 이걸 두 종류의 RNN을 encoder 파트와 decoder 파트로 구분하여 이어 붙이는 방식으로 구현했습니다. 아래의 이미지를 한 번 보시죠. 

<div style="text-align:center;">
        <img src="_static/figures/seq2seq을 위한 rnn 활용.jpg" width="950">
</div>

자 이 그림을 보면 Encoder 파트와 Decoder 파트로 나뉘어 있습니다. 그리고 각 파트를 RNN 블럭으로 구성하고 있습니다. RNN 블럭은 방금 배운 LSTM 블럭으로 대체해도 됩니다. LSTM도 넓게 보면 RNN에 들어가기 때문에, 여기서는 RNN이라고 표기하겠습니다. 

이런 형상은 seq2seq 모델이 하는 일을 생각해보면 자연스럽게 이해가 됩니다. 일단 seq2seq 모델은 입력과 출력의 길이가 다를 수 있고, 게다가 데이터마다 길이가 제각각입니다. 그래서 두 개의 RNN 모델이 필요합니다. 하나는 입력되는 가변적인 데이터에 필요한 RNN 모델입니다. 다른 하나는 가변적인 길이의 출력을 담당할 RNN 모델입니다. 

이런 방식은 시퀀스 구성 요소들 사이의 1대 1 대응을 벗어나 입력 시퀀스의 전반적인 내용에 기반하여 적절한 출력 시퀀스를 만들어야 하는 seq2seq에 잘 맞습니다. 앞서 설명드렸듯이, 번역이든 챗봇이든, 입출력의 시퀀스 구성요소들은 1대 1 대응 관계가 명백하지 않습니다. 길이도 보통 다릅니다. 그러니 첫째, 입력 정보를 순서 고려하여 적절한 수치 표현으로 전환하고, 둘째, 이 수치 표현을 참조해서 적절하게 출력을 만들어야 합니다. 전자는 Encoder, 후자를 Decoder가 하는 셈이지요. 

특히 기계 번역에서 이런 디자인이 많이 활용됩니다. 기계 번역에서 이 디자인을 활용한 신경망 모델 자체도 상당히 중요한 혁신이었다고 합니다. 저도 기계 번역의 역사는 잘 모르지만, 그 전에는 시퀀스를 부분 하위 요소로 쪼개서 출력에 1대 1 대응시켜주는 방식으로 모델링 하는 일이 많았다고 하는데, 이렇게 전체 시퀀스의 표현을 만들고 그걸 참조해서 출력을 뽑아내는 신경망 모델이 기계 번역에서 큰 성공을 거뒀다고 하네요. 

이 그림 역시 기계 번역 사례를 표현하고 있습니다. <나는>, <학생>, <이다> 가 I, AM, A, STUDENT로 전환되는 seq2seq 사례죠. 인코더 파트는 앞에서 배웠던 것과 완전히 같습니다. '나는 학생 이다'가 순차적으로 입력되면서 은닉 상태가 갱신됩니다. 마지막 은닉 상태가 $H_4$입니다. 원리상, 여기에는 '나는 학생 이다'에 대한 정보가 다 압축되어 있지요. 

디코더 파트에서는 이 $H_4$를 인코더에서 들어온 전체 맥락, C로 삼고, 그걸 매 계산마다 고려합니다. 여기서 C가 context를 줄여서 만든 기호임을 기억하십시오. 앞 LSTM에서는 Cell state를 줄여서 C라고 표현했는데, 여기서는 다른 것입니다. 다른 문자로 적을까 하다가, Context라는 어감을 살리기 위해 그냥 C라고 쓰기로 결정했으니, 혼동 없으시기 바랍니다.  

자 다시 돌아와서 디코더 RNN 블럭은 세 가지 정보를 종합해서 특정 시점 은닉 상태를 개선합니다. 세 종류의 정보란, 첫째, 이전 시점 은닉 상태, 둘째, 이전 시점 출력, 셋째, 인코더에서 넘어온 맥락을 의미합니다. 그리고 그 은닉 상태를 기반으로 해당 시점 출력을 만듭니다. 

예를 들어 'AM'이라는 출력이 만들어지는 과정은 두 단계를 거칩니다. 일단 $t_1$ 시점에서의 은닉 상태, $S_1$을 만들어야 합니다. 원래 H를 은닉 상태의 약자로 썼는데, 디코더를 인코더와 구분하기 위해서 여기는 S를 쓰겠습니다. 그리고 그걸 다시 $Y_0$, $C$와 함께 연산하여 $Y_1$ 즉 'AM'을 만듭니다. 이걸 수식으로 정리하면 아래와 같습니다. 


$$
\begin{aligned}
S_1 &= f(S_0, Y_0, C)\\[15pt]
P(Y_1|Y_0, X) &= g(S_1, Y_0, C)
\end{aligned}
$$

이게 디코더 전체 시퀀스에서 일어난다고 보면 됩니다. 즉 일반화하면 다음과 같은 형태인거죠. 

$$
\begin{aligned}
S_t &= f(S_{t-1}, Y_{t-1}, C)\\[15pt]
P(Y_t|Y_{<t}, X) &= g(S_t, Y_{t-1}, C)
\end{aligned}
$$


여기서 f는 RNN에서 일어나는 연산을 가리키는 함수이고, g는 출력 변환을 위한 함수라고 보시면 됩니다. 약간 의아하실 수 있는 포인트가 세 가지 있습니다. 간단히 살펴볼게요. 

우선 원래 RNN에는 두 개의 요소가 입력되었는데 여기서는 세 개입니다. 즉 '이전 시점 은닉 상태'와 '현재 시점 입력 자료'가 들어갔죠. 그런데 디코더 RNN에서는 이전 시점 은닉 상태는 $S_{t-1}$로, 현재 시점 입력 자료는 $Y_{t-1}$로 들어가고 있는데, C가 하나 더 붙었네요. 그런데 이건 쉽게 해결할 수 있습니다. 예를 들어서 $tanh(W_s S_{t-1} + W_y Y_{t-1}+ W_c C + b)$ 이런식으로 새로운 은닉 상태를 만들면 되죠. 이거 말고도 여러 방식이 가능합니다. 요컨대, 여러 맥락 정보와 지금 시점의 입력을 종합하는 방식은 다양할 수 있으니 그걸 쓰고 있다고 보시면 됩니다. 

두 번째로, 현재 입력 정보가 왜 $Y_{t-1}$ 으로 표현되고 있을까요? 이건 번역 작업을 포함하여 언어 생성 작업의 특성 때문입니다. 보통 이럴 경우 입력 자료가 '바로 전 시점의 토큰'이 되는 경우가 많습니다. 그래서 전 시점에 출력된 결과인 토큰을 쓰고 있는거죠. (여담이지만, 전 시점에 모델이 틀린 토큰을 출력했을 가능성이 있기에, 보통 학습 과정에서는 이 경우 '정답'을 넣어줍니다. 즉 이 경우 I를 앞에서 제대로 출력하지 못했다 해도, I를 넣어준다는 말입니다. 이걸 teacher forcing이라고 합니다) 

세 번째로, g는 어디서 튀어 나왔나 싶으실겁니다. 그런데 앞에서 부터 RNN이 매 시점 뭔가 유의미한 출력을 만들어야 할 때, 그 시점에 만들어진 은닉 상태에 기반하여 출력물을 만드는 층(layer)을 따로 건다고 말씀드렸습니다. 출력층을 따로 마련한다는 말은, H에 기반한 연산을 수행한다는 뜻입니다. 지금은 g가 그 역할을 하고 있습니다. 이 g 역시 다양한 방식으로 수행될 수 있습니다. 세 개의 벡터를 이어서 (concatenate) 완전 연결 신경망을 통과한 후 softmax를 거는 방식이 가장 쉽게 떠올려 볼 수 있는 방식이겠네요. 여기서는 그저 어떤 연산을 통과한다는 뜻으로, 그림에는 '출력 변환 g'라고 표기하고, g라는 함수를 통과한다고만 해두겠습니다. 그림에서는 하늘색 박스로 표시하고 있습니다. (그래서 사실 그림에서 이 하늘색 박스에 두 개의 선이 더 들어가야 하지만, 그림이 알아보기 힘들게 복잡해져서 생략했습니다.)



자 이제 대략 전체가 이해가 되실거라 생각합니다. 참 괜찮은 아이디어죠. 그런데 여기서 다시 한 번 이 RNN 기반 모델 디자인이 가진 특징에 주목해보죠. 특히 Encoder의 정보가 Decoder에 전달되는 과정에 주목해봅시다. 지금 그림을 보면, 그건 순전히 $H_4$라는 마지막 은닉 상태에 기대고 있습니다. 그게 Encoder에 입력된 문장 전체를 잘 요약해서 전달하고 있는 셈입니다. 그리고 그게 매 순간 출력마다 맥락으로 개입하고 있습니다. 주황색 선을 보면, 이 은닉 상태가 Decoder RNN 블럭에 매번 전달되고 있습니다. 물론 이 은닉 상태를 만들기 위해 뭔가 처리를 더할 수는 있습니다. 예를 들어 $H_{final} = f(H_0, H_1, H_2, H_3, H_4)$로 따로 만들 수도 있죠. 하지만 뭐가 되었든 마지막 은닉 상태를 만들어서 전달하는 것이 일반적인 디자인입니다. 

그런데 이런 디자인은 한계가 명확하다고도 볼 수 있습니다. 왜냐하면, **앞서 말했듯이 정해진 길이의 벡터 안에 encoder에 입력된 전체 정보를 압축하려고 하고 있기 때문입니다.** 물론 LSTM 같은 계량된 RNN을 쓰면 약간 문제가 완화될 것입니다. 예를 들어 C와 H를 함께 쓰면, 바닐라 RNN보다는 낫겠지요. 하지만 여러번 지적했듯이 이 경우에도 고정 길이 벡터라는 점에는 변화가 없습니다. 그 벡터의 길이가 좀 더 길어졌고 내용이 섬세하게 만들어져 있을 뿐입니다. 이게 왜 문제가 될까요? 앞서 말한 것처럼 정보의 겹침과 약화 문제가 생기기 때문입니다. Decoder에서는 이 마지막 산출 고정 길이 벡터를 갖고 모든 일을 다 해야 합니다. 이 벡터에 앞선 시퀀스의 모든 정보가 거기 들어가 있기를 기대합니다. 그런데, 만약 입력 시퀀스가 엄청 길어지면 어떨까요? 그러면 마치 도화지에 여러 개의 그림을 그린 것처럼, 정보가 겹치고 그 때문에 정보의 가치가 떨어져 있을 가능성이 큽니다.

앞서 말한 기념비적 논문을 작성한 바다나우 등도 이것이 문제를 낳을 수 있다고 추측했습니다. **고정 길이 벡터 (fixed length vector)로 인코더의 정보를 디코더로 전달하는 것이 핵심적인 병목 (bottleneck)** 이 될 수 있다고 추측한거죠. 병목을 증명하거나 관찰하는 것은 어려운 일이겠지만, 기계 번역에서 문장 길이가 길어질 수록 품질이 떨어졌다고 합니다. 이건 고정 길이 벡터가 정보 전달의 병목이 될 수 있다는 중요한 증거였습니다. 

그러면 어떻게 하면 좋을까요? 바다나우 등이 제시한 방법은 이러했습니다. 인코더의 은닉 상태를 마지막 버전만 활용하는 대신, 그냥 앞에 있는거 다 가져와서 활용하는 것입니다. 즉 위의 그림으로 보자면, $H_4$ 만 활용하는게 아니라, $H_0, H_1, H_2, H_3, H_4$를 다 활용한다는 말입니다. 나아가 다 활용하되, 매 시점마다 초점을 둬서 활용합니다. 

디코더 입장에서 매 시퀀스를 출력할 때 인코더에 입력된 정보가 잘 보존되어서 전달되면 좋지만, 그 모든 정보가 매 시퀀스마다 동등하게 필요한건 아닐 수 있습니다. 기계 번역 상황을 생각해보죠. 그림의 예처럼 '나는 학생이다'를 'I am a student'로 번역하는 상황을 생각해봅시다. 그러면 I를 출력해야 할 때 인코더 쪽에서 가장 고려해야 할 정보는 뭘까요? 아마도 '나는'이 입력된 시점에 만들어진 은닉 상태, 즉 $H_1$일 것입니다. 반대로 student를 출력할 때는 어떨까요? 그러면 그건 아마 '학생'이 입력된 시점에 만들어진 은닉 상태, 즉 $H_2$일 것입니다. 

그래서 이런 방식으로 활용하는 것입니다. 원래는 디코더는 모든 출력에서 $H_4$, 즉 C를 참조로 했습니다. 즉 디코더 RNN 모델의 이전 은닉 상태($S_{t-1}$)와 지금 입력 자료 ($Y_{t-1}$), 그리고 C를 참조로 해서 모든 단어를 생산해야 했죠. 그런데 그게 이제 이렇게 바뀝니다. 

'I'를 만들 때에는 C, 즉 $H_4$ 대신 $0.01 \cdot H_0 + 0.96 \cdot H_1 + 0.01 \cdot H_2 + 0.01 \cdot H_3 + 0.01 \cdot H_4$ 이라는 벡터를 만들어서 참조합니다. 그리고 'Student'를 만들 때에는 $0.01 \cdot H_0 + 0.01 \cdot H_1 + 0.96 \cdot H_2 + 0.01 \cdot H_3 + 0.01 \cdot H_4$ 이라는 벡터를 만들어서 참조합니다. 

요컨대 바다나우 어텐션의 핵심은 이렇습니다. 인코더의 은닉 상태 H 들을 다 가져옵니다. 그리고 디코더 출력마다 이를 적합하게 가중 평균하여 맥락으로 활용합니다. 이렇게 함으로써, 하나의 고정 길이 벡터에 인코더 정보를 모두 집어넣어야 하는 제한을 넘어섭니다. 동시에 전체 정보를 무조건 다 활용하는게 아니라 그때그때 집중점을 만들면서 종합해서 활용하는 거죠. **즉 이 장치는, 매 순간 입력 시퀀스에서 기인한 정보 중 어떤 면에 주목을 기울여야 하는지 살펴보는 메커니즘, 즉 '주목'(attention) 메커니즘이라고 부를 수 있는 것입니다.**




이걸 수식화해서 표현하면 다음과 같습니다. 

$$
\begin{aligned}
S_t &= f(S_{t-1}, Y_{t-1}, C_t)\\[15pt]
P(Y_t|Y_{<t}, X) &= g(S_t, Y_{t-1}, C_t)
\end{aligned}
$$


변한건, $C$가 $C_t$가 된 것 밖에 없습니다. 그런데 이게 핵심 변화입니다. 원래는 인코더의 모든 정보가 C라는 하나의 변치 않는 벡터로 만들어져 디코더에서 참조되었습니다. 그러니까, 1) 그 C가 너무 긴 입력 시퀀스 때문에 정보의 겹침과 약화에 노출될 가능성과 2) 매 순간마다 전체 입력 정보를 다 활용할 수 밖에 없는 한계에 노출되어 있었던 것이지요. 그런데 바다나우 어텐션은 매 시점마다 맥락 정보 $C_t$를 계산합니다. 그리고 앞서 말한 것처럼 $C_t$는 H들의 가중 평균입니다. 그러면 이 가중평균을 어떻게 만드냐가 다시 핵심이 되겠네요. 수식으로 표현하면 아래와 같습니다. 

$$
\begin{aligned}
C_t &= \sum_{i=1}^{T_x} \alpha_{ti}H_i  \\[15pt]
\alpha_{ti} &= \frac{exp(e_{ti})}{\sum_{k=1}^{T_x}exp(e_{tk})}  \\[15pt]
e_{ti} &= a(S_{t-1}, H_i)
\end{aligned}
$$


뭔 소리지 싶으실 수 있는데, 별거 아닙니다. 일단 $C_t$는 H들을 다 가중 평균해서 만듭니다. 즉 t 시점의 $\alpha$ 세트를 H들에 적절히 곱해서 다 더하는거죠. 그러면 이 $\alpha$ 세트는 어디에서 오는가? 그건 $e$ 세트들을 만들어서 그걸 소프트맥스 함수에 통과시켜 만듭니다. 즉 $e$라는 기초 수치를 만들어서 확률 분포로 전환하는 거죠. 그러면 이 $e$들은 어떻게 만들까요? 이건 매 시점마다 계산이 됩니다. 그리고 그건, 디코더에 과거 시점 은닉 상태 $S_{t-1}$과 인코더 은닉 상태 H들의 연산에 기반합니다. 그게 $a$라는 함수가 가리키는 부분입니다. 

자 이 $a$는 여러 가지 형태로 구현될 수 있습니다. 바다나우 어텐션 논문에서는 이걸 별도의 신경망으로 구현합니다. 그런데 그 경우 약간 설명이 복잡하니 이후에도 많이 쓰는 dot product를 활용해서 구현한다고 단순화하여 가정하겠습니다. 이 경우는 $e_{ti}$는 아래와 같이 계산되겠죠. $S_{t-1}, H_i$두 벡터의 차원이 같다고 가정하겠습니다. 

$$
\begin{aligned}
e_{ti} = S_{t-1} \cdot H_i
\end{aligned}
$$

간단하지만 이것도 충분히 말이 되는 과정입니다. 왜냐하면, dot product는 벡터 사이의 유사성을 보여주는 기초 지표가 될 수 있기 때문입니다. dot product는 길이가 같은 벡터 사이에서 같은 위치에 있는 구성 요소들을 곱해서 다 더하는 것인데요, 그러면 이 숫자가 커진다는 말은 같은 위치에 있는 숫자들이 방향이 대체적으로 같고, 둘이 곱한 값이 대체적으로 컸다는 말이죠. 그러면 dot product 결과가 크면, 벡터의 방향과 강도가 유사할 가능성이 큽니다. 물론 이 결과는 벡터의 길이에도 영향을 받습니다. 그래서 각 벡터의 길이로 나눠주곤 하는데요, 그게 벡터 사이의 유사성을 보는 cosine similarity입니다. 

다시 돌아와서, 이런 방식으로 어텐션 메커니즘을 도입하면 뭐가 좋을까요? 여기서 매 시점 $\alpha$들이 분석자가 입력하는게 아니라 모델 학습에 근거해서 조절될 수 있는 수치라는 점을 기억해야 합니다. 바다나우 어텐션 세팅에 따르면 아예 명시적으로 이 점수를 계산하는 별개의 신경망 파라미터가 학습되고, 위에 제가 제시한 버전에서는 S와 H들이 주목을 고려해서 학습되겠지요. 요컨대, seq2seq 모델이 잘 기능하도록 $\alpha$들이 학습될 수 있다는 말입니다. 그러면, 이 모델은 '디코더에서 출력을 만드는 매 순간마다 인코더 은닉 상태 중 어디에 주목해야 할지 학습하여 적용하는' 신경망 모델이 됩니다. 대박이죠. 앞서 말했듯 실제로 매우 큰 성능 증가가 있었습니다. 고정 길이 벡터라는 일종의 병목도 해결하고, 유연하게 주목을 조절할 수 있었으니까요. 

지금까지의 조치를 그림으로 표현하면 아래와 같습니다. 즉, attention을 부착한 seq2seq 모델이죠. 어텐션 메커니즘이 일종의 **동적 맥락 생성기**에 가까워서, 모든 과정을 그림으로 표현하기 어려운 면이 있습니다. 그래서 아래에서는 $C_1$ 즉 디코더에서 두 번째 연산의 맥락 정보를 만드는 것만을 그림으로 표현했습니다. 

<div style="text-align:center;">
        <img src="_static/figures/seq2seq을 위한 rnn 활용 어텐션 부착.jpg" width="950">
</div>

자 이제 간단히 보이실 겁니다. 디코더 두 번째 시점 연산이 들어갈 때, 그 전 시점에 디코더에서 생산된 은닉 상태($S_0$)과 인코더 은닉 상태 간의 연산에 기반하여, 인코더 은닉 상태들의 가중 평균값이 만들어집니다. 그게 $C_1$입니다. 그리고 그게 두 번째 연산에 개입합니다. 두 번째 디코더 은닉 상태 ($S_1$)를 만드는 것과 두 번째 출력 ($Y_1$)을 만드는데 개입하죠. 이 과정을 모든 단계마다 반복하는 것입니다. 그때그때 맥락값을 동적으로 만들어서 반응하는 거죠. 그리고 여러 번 말하지만 이렇게 하는 까닭은, 인코더 은닉 상태 중 어디에 주목할지 유연하게 동적으로 정하기 위함입니다. 

이처럼 유연한 주목점을 인코더 은닉층을 가중 평균하는 방식으로 만들어서 연산을 진행할 때 생기는 장점에 대한 직관적인 예가 기계 번역에는 아주 많이 있습니다. 바다나우 어텐션 논문 6페이지에 그런게 많이 나오죠. 저작권 문제로 그냥 설명으로 전하자면^^;; 예를 들어 이런 어텐션 메커니즘은 언어 마다 어순이 달라 생기는 일종의 정렬 문제를 효과적으로 극복합니다. 예를 들어 L' accord sur la zone economique europeenne a ete signe an aout 1992. 라는 프랑스어 문장을 The agreement on the European Economic Area was signed in August 1992. 라는 문장으로 옮겨야 한다고 가정해봅시다. 일부 토큰은 순서대로 집중적으로 인코더 측 은닉 상태를 참조하면서 옮기면 되지만 (예를 들어 agreement를 출력할 때에는 accord에 주목하는 식이죠) 일부는 그렇지 않습니다. 어순 문제가 대표적입니다. 예를 들어 유럽 경제 구역을 영어에서는 European Economic Area 라고 하는데 프랑스어에서는 zone economique europeenne 라고 하네요. 정확히 거꾸로인거죠. 

그런데 이런 어텐션 메커니즘을 활용하여 모델이 학습되면, European을 출력할 때에는 기계적으로 인코더에 입력된 다섯 번째 토큰인 zone을 보는게 아니라 europeenne를 보게 만들 수 있습니다. 그래야 성능이 올라갈 수 있으니까요. 그래서 번역이 잘 되죠. 뿐만 아닙니다. 지금 우리가 설명한 장치가 부드러운 주목을 가능케 함도 중요합니다. 즉 꼭 인코더에서 하나의 은닉 상태를 불연속적으로 골라서 주목하는게 아니라는 말입니다. 여러 개에 동시 주목을 할 수 있죠. 바다나우 등이 쓴 논문에서는 이 기능에 대한 예로 [the man]을 [l'homme] 로 옮기는 사례를 예로 듭니다. 프랑스어에서는 정관사가 변형이 많다고 하네요. 단어에 따라 le/la/les/l' 등 다양한 버전이 붙을 수 있다고 합니다. 그 경우 the를 제대로 옮기려면 man까지 봐야죠. 이 경우 어텐션 메커니즘은 이런 식으로 작동할 수 있습니다. l'를 번역하는 시점에서 0.5*(the가 입력된 시점에 만들어진 은닉상태) + 0.4*(man이 입력된 시점에 만들어진 은닉상태)+... 를 C로 만들어서 연산에 개입시키는거죠. 

이런 어텐션 메커니즘은 꼭 기계 번역 분만 아니라 seq2seq 모델로 표현되는 다양한 과제에서 아주 큰 성능 증가를 가져왔습니다. 왜냐하면, 1) 디코더의 출력마다 인코더에서 산출된 은닉 상태들 중 어디에 집중적으로 주목하면 좋을지 연산이기에, 고정 벡터로 정보를 입력 정보를 압축하는 한계를 넘어섭니다. 2) 이 주목이 부드럽게 이루어지기에, 1:1 대응을 넘어서 있는 관계 포착을 자연스럽게 수행합니다. 

그리고 이 강의 말미에 설명하겠지만, 어텐션이라는 아이디어는 seq2seq의 개선을 넘어 더 큰 변화의 씨앗이 되었습니다. RNN을 대체할 잠재성을 가진 아키텍쳐의 씨앗이 된거죠. 

## 5. 남은 한계, 그리고 Transformer

자 그러면 이제 RNN의 모든 문제가 해결되었나요? 예상하시겠지만 그렇지 않습니다. 우리가 3장을 시작할 때 RNN이 두 가지 한계가 있다고 지적했습니다. 1) 정보의 중첩과 약화, 2) 순서를 반영한 RNN 연산의 기계적 비효율이 그것이었습니다. 지금까지 LSTM과 Attention을 거쳐서 완화된 것은 첫 번째 한계였습니다. 그럼 두 번째 한계는 어떤가요? 이건 여전히 남아있습니다. 

RNN의 핵심은 시퀀스 데이터가 가진 순서 구조 정보를 살리겠다는 것이었습니다. 이건 LSTM이 되든 Seq2seq이 되든 마찬가지입니다. 그래서 시퀀스를 구성하는 요소들이 순서대로 계산됩니다. 앞쪽 구성요소의 계산 결과가 뒤쪽 구성요소의 계산 결과에 영향을 미치도록 설계되어 있습니다. 이건 합당한 조치입니다. 앞서 말했듯 언어 같은 정보는 순서가 중요하고, 그것을 고려해서 정보를 처리해야 할 필요가 있기 때문입니다. 

그런데, 이런 구조는 연산이라는 관점에서는 상당한 비효율을 만들 수 있습니다. 왜냐하면, 아무리 기계의 연산 능력이 좋아도, 그래서 동시에 여러 연산을 할 수 있어도, RNN 구조에서 시퀀스 자료를 처리할 때에는 하나하나 순서대로 계산을 해야 합니다. 예를 들어 [a,b,c,d]라는 시퀀스를 RNN으로 처리한다고 하죠. 우리의 기계가 네 개의 간단한 연산을 동시에 할 수 있는 능력이 있다고 합시다. 그러면 a,b,c,d를 동시에 입력해서 연산할 수 있나요? 그렇지 않습니다. 왜냐하면, b라는 입력을 처리하기 위해서는 a라는 입력을 처리한 결과가 은닉 상태로 넘어와 줘야 합니다. c를 처리하기 위해서는 b까지 처리된 결과가 필요하고, d를 처리하기 위해서는 c까지 처리된 결과가 필요하죠. 즉, 기다려야 한다는 말입니다. 

앞서 말했듯이, 이건 순서 정보를 살리기 위한 선택이었습니다. 대신 연산 비효율을 감수해야 하는 상황인거죠. 그런데 이 단점은 모델이 복잡해지고 학습 데이터가 늘어날수록 생각보다 큰 문제가 되었습니다. 학습에 너무 오랜 시간이 걸리게 된 것입니다. 구조 자체가 병목으로 기능하는 상황이 만들어진 것이죠. 

그런데 앞서 살펴본 attention이라는 아이디어로부터 이를 극복하는 발상이 나왔습니다. 유명한 트랜스포머 (Transformer) 라는 아키텍쳐가 그것입니다. 2017년 이 아키텍쳐가 제안된 이후, 많은 영역에서 RNN이 하던 일을 트랜스포머가 대체하기 시작했습니다. 이 새로운 딥러닝 아키텍쳐는 병렬 연산을 적극 활용하면서도 시퀀스가 가진 순서 정보를 참고 및 활용하고 있습니다. 정확히 말하면 RNN 만큼 순서 정보를 절대적 구조로 상정하지 않지만, 훨씬 유연하고 필요한 만큼 활용합니다. 

이게 어떻게 가능했을까요? 이 트랜스포머 아키텍쳐가 제안된 2017년의 기념비적 논문의 제목은 'Attention Is All You Need'입니다. 즉 어텐션만 있으면 충분하다. 어텐션으로 RNN이 하던 일을 상당 부분 해낼 수 있다는 도발적 메시지를 담고 있습니다. 

어떻게 이게 가능할까요? 이건 어텐션을 seq2seq을 개선하는데만 쓰지 않고, **아예 처음부터 입력된 시퀀스의 구성요소를 처리할 때 함께 입력되는 다른 구성요소를 참조하는 메커니즘으로 쓰면서 가능해졌습니다.** 앞선 절에서 어텐션은 무슨 일을 했나요? 인코더의 은닉 상태들을 동적으로 참조해서 결과를 만들어내는 일을 했습니다. 즉 어텐션은, 앞에서 주어진 문제 상황과 분리해보면, **참조 대상이 되는 여러 가지 요소들을 과제와 목표에 따라 유연하게 참조하게 만드는 메커니즘**인 셈입니다. 그런데 이 메커니즘을 하나의 시퀀스 구성요소를 처리할 때, 즉 그것을 가지고 어떤 연산을 수행할 때, 해당 시퀀스를 구성하는 다른 요소들을 참조하게 만드는 방식으로 쓸 수 있습니다. 즉 참조 대상을 인코더 쪽 은닉 상태가 아니라, 시퀀스를 구성하는 다른 요소들로 삼겠다는 말입니다. 

RNN이 연산에서 순서 구조를 유지한 까닭은, 시퀀스를 이루는 구성요소들을 연산에 투입할 때 그보다 먼저 존재하는 구성요소들을 고려하도록 만들기 위함이었습니다. 앞서 말한 것처럼 [a,b,c,d]라는 시퀀스가 있다면, d를 처리할 때 a,b,c의 결과를 고려하게 만들고 싶었던거죠. 그런데 어텐션의 참조 대상을 시퀀스 자체로 삼으면, 이게 다른 방식으로 가능해집니다. 예를 들어 d를 처리할 때, 그걸 위한 C를 만들어서, 예를 들어 $C_d = a 변환 값 \cdot 0.5 + b 변환 값 \cdot 0.4 + c 변환 값 \cdot 0.05 + d 변환 값 \cdot 0.05$ 이런 값을 만들어서 참조하는 것입니다. 게다가 각 구성요소들의 변환값에 실리는 가중치가 학습 가능하다고 하다면, RNN보다 훨씬 유연하죠.

이런 유연한 상호 참조 메커니즘은 RNN의 고전적인 문제인 먼거리 상호작용 포착 어려움 문제를 완화하는 효과도 가집니다. RNN이 순서대로 연산하기라는 구조를 가지고 있던 것은, 결국 시퀀스 구성요소들의 상호의존을 고려하기 위함이었다고 해석할 수 있습니다. 상호의존에서 '순서'가 그 상호의존의 강도와 방향을 결정하는 매우 중요한 요소라고 생각했던 것이지요. 순서상 가까이 붙어 있으면 아무래도 상호의존이 좀 더 강하게 고려되는 구조적 편향을 가진 것도 그 때문일 것입니다. 언어 사례에서 보듯, 이는 매우 설득력이 있는 면이 있습니다. 하지만 절대적이지는 않습니다. 앞서 설명한 것처럼 순서를 고려하면 매우 멀리 떨어져 있지만, 강한 상호의존을 가지고 작동하는 언어적 작용도 드물지 않으니까요. 

어텐션을 애초에 시퀀스 구성요소의 기초 연산부터 활용하게 되면, 이 부분이, 즉 시퀀스 구성요소들이 꽤 거리를 두고 강하게 의존하는 패턴의 포착이 상대적으로 쉬워집니다. RNN에게 순서는 매우 절대적인 정보이지만, 어텐션은 특별히 그걸 신경쓰지 않습니다. 앞선 예에서 a, b, c에 가중치를 붙일 때, 특별히 d와 가까이 있는지 그렇지 않은지 고려하지 않을 수 있다는 말입니다. 멀리 떨어져 있는 것에도 큰 가중치를 붙일 수 있는 가능성이 열려 있지요. 

물론 이렇게 완전히 순서에서 자유롭게 만들어두면, 거꾸로 순서가 주는 정보를 놓치는 면이 생길 것입니다. 너무 과하게 절대적인 구조라고 여기는게 아쉬울 뿐, 여전히 순서는 중요한 정보입니다. 그래서 트랜스포머에서 어텐션을 활용할 때, 각 구성요소의 위치를 따로 정보로 입력해주는 방식을 택했습니다. 위치 인코딩 (positional encoding)이라는 기법이 그것입니다. 자세한 것은 다음 장에 살펴보지요. 

그리고 결정적으로 이렇게 어텐션을 활용해서 시퀀스 구성요소들의 상호 의존을 고려한 연산을 수행하면, RNN의 중요한 한계인 연산 비효율을 넘어설 가능성이 생깁니다. 아주 간단히 설명하자면 이렇습니다. 앞서 언급한 것처럼 [a,b,c,d]라는 시퀀스를 처리한다고 생각해봅시다. RNN은 d의 기본 수치 표현을 모델의 목적에 맞게 변환할 때, d보다 앞서 있는 a, b, c의 표현이 처리되어 그 결과가 넘어오기를 기다려야 합니다. 반면 위처럼 어텐션을 활용하면, 앞선 구성요소 연산 결과를 순서대로 기다릴 것 없이 여러 위치에서 동시에 연산을 수행할 수 있습니다. 즉 병렬 연산이 가능하다는 말입니다. 이러면 GPU 같은 기계의 성능을 훨씬 적극적으로 활용할 수 있죠. 역시 자세한 사항은 다음 장에 살펴보겠습니다. 

요컨대, RNN은 여러 개선에도 불구하고 연산 비효율이라는 한계를 가지고 있었습니다. 그런데, RNN의 한계를 극복하려고 제안되었던 attention이라는 메커니즘이, 아예 RNN과 다른 방식으로 시퀀스 데이터를 처리하는 방법의 씨앗이 되었습니다. 그 결과가 트랜스포머 아키텍쳐입니다. 다음 장에서 이를 살펴보겠습니다. 